# Phase 3 — Exploratory Data Analysis
## Patterns of Recorded Civilian Targeting in Gaza

**DS312 · Group 7 · Final merged Phase 3 notebook**

This notebook merges the team's Phase 3 contributions in the task-tracker order: **(1) univariate analysis, (2) bivariate and multivariate analysis, (3) visualization, (4) hypothesis testing, (5) pattern spotting, (6) business-question answers, and (7) modeling implications.**

It uses the Phase 2 processed CSVs, loaded once below. Each section states the data unit and denominator. Figures stay beside the analysis they explain, and the visualization section indexes the figure set for rubric review.

> Findings describe recorded ACLED events and the project's focal-event rule. A flag, association, cluster, or anomaly does not establish intent, responsibility, causation, or legal classification.

## Phase 3 rubric map

| Guide task | Notebook section | Evidence |
|---|---|---|
| Univariate distributions, summaries, frequencies | Task 1 | Profile tables, numeric summaries, category counts and plots |
| Bivariate/multivariate relationships and group comparisons | Task 2 | Period, event type, severity, precision, actor, district-week, and weekly comparisons |
| At least 8 titled, labeled, interpreted visuals | Task 3 chart register; figures beside Tasks 1, 2, 4, 5 | Register identifies over 8 purposeful charts |
| At least 2 statistical hypotheses, p-values, conclusions | Task 4 | Two tests, sparse-cell checks, effect sizes, conclusions |
| Segments, trends, seasonality, suspicious values, imbalance | Task 5 | Pattern analyses and register |
| Answer/refine Phase 1 business questions | Task 6 | BQ1-BQ5 answers reconciled to the Phase 2 CSVs |
| Modeling implications | Task 7 | Candidate features, methods, validation, and risks |

Every chart should have a question-led title, labeled axes/legend, and a 1-2 sentence interpretation. Keep the interpretation directly after its chart.

## Phase 2 CSV map and units

| CSV in `data/processed/` | One row represents | Phase 3 use | Scope note |
|---|---|---|---|
| `acled_events_clean.csv` | One Gaza event, both periods | Event-level distributions, before/after, actor and event profiles | All Gaza-coded events retained; four have no district. |
| `acled_gaza_week.csv` | One Gaza-wide week after Oct 7, 2023 | Weekly trends, coverage, history features | Zero-event weeks retained; last week is partial; zero-denominator share is undefined. |
| `acled_district_week.csv` | One known district-week after Oct 7 | District comparisons, persistence, anomalies | Four records without a district are omitted only here. |
| `acled_focal_incidents.csv` | One post-Oct 7 land focal event | Spatial views, precision sensitivity, DBSCAN review | Coast-coded events excluded only for land spatial analysis; repeated coordinates remain. |

Run from the project folder, or set `DATA_DIR` in the loader if the files are elsewhere.

In [ ]:
# Set thread controls before scikit-learn imports (avoids Windows worker warnings).
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
import warnings
warnings.filterwarnings("ignore", message=".*physical cores.*")
warnings.filterwarnings("ignore", message=".*memory leak on Windows.*")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy.stats import chi2_contingency, MonteCarloMethod, binomtest, fisher_exact, spearmanr
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10, "legend.fontsize": 9})
plt.rcParams["axes.grid.axis"] = "y"
SEED, ALPHA, N_RESAMPLES, Z = 42, 0.05, 99_999, 1.96
np.random.seed(SEED)
C_FOCAL, C_OTHER, C_GREY, C_WARN = "#8D3B42", "#315A7D", "#9AA5B1", "#E0A458"

def find_data_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "data" / "processed"
        if (candidate / "acled_events_clean.csv").is_file(): return candidate
    raise FileNotFoundError("Cannot find data/processed/acled_events_clean.csv. Run from the project root or set DATA_DIR.")

DATA_DIR = find_data_dir()
master = pd.read_csv(DATA_DIR / "acled_events_clean.csv", parse_dates=["event_date", "month_start", "week_start"])
for col in ("geo_precision", "time_precision"):
    master[col] = pd.Categorical(master[col], categories=[1, 2, 3], ordered=True)
gaza_week = pd.read_csv(DATA_DIR / "acled_gaza_week.csv", parse_dates=["week_start", "week_end"])
district_week = pd.read_csv(DATA_DIR / "acled_district_week.csv", parse_dates=["week_start", "week_end"])
incidents = pd.read_csv(DATA_DIR / "acled_focal_incidents.csv", parse_dates=["event_date", "month_start", "week_start"])
for col in ("geo_precision", "time_precision"):
    incidents[col] = pd.Categorical(incidents[col], categories=[1, 2, 3], ordered=True)
# Aliases retain the names used by the separate contribution notebooks.
events, gaza_weekly, district_weekly = master, gaza_week, district_week
BEFORE, AFTER = "Before Oct 7, 2023", "Oct 7, 2023 onward"
CUTOFF = pd.Timestamp("2023-10-07")
COVERAGE_START, COVERAGE_END = master.event_date.min(), master.event_date.max()
PERIOD_DAYS = {BEFORE: (CUTOFF-COVERAGE_START).days, AFTER: (COVERAGE_END-CUTOFF).days+1}
EVENT_TYPES = ["Explosions/Remote violence", "Violence against civilians", "Battles", "Protests", "Riots"]
political = master.loc[master.political_violence].copy()
print("Phase 2 directory:", DATA_DIR.resolve())
print("master:",master.shape,"| gaza_week:",gaza_week.shape,"| district_week:",district_week.shape,"| incidents:",incidents.shape)
print("Date coverage:",COVERAGE_START.date(),"to",COVERAGE_END.date())

### Shared Phase 2 integrity check

Confirm the loaded tables against the Phase 2 snapshot before using saved interpretations. A mismatch means the source data or scope needs to be reconciled.

In [ ]:
checks = pd.DataFrame([
 ("Event master",(26664,43),master.shape),("Gaza-week",(104,13),gaza_week.shape),
 ("District-week",(520,14),district_week.shape),("Land focal incidents",(10583,45),incidents.shape),
 ("Focal events",10714,int(master.focal_event.sum())),
 ("Focal before cutoff",63,int(master.loc[master.period==BEFORE,"focal_event"].sum())),
 ("Focal from cutoff",10651,int(master.loc[master.period==AFTER,"focal_event"].sum())),
 ("Political records before",389,int((political.period==BEFORE).sum())),
 ("Political records from cutoff",22364,int((political.period==AFTER).sum())),
 ("Missing districts",4,int(master.admin2.isna().sum())),
 ("Gaza-week focal total",10651,int(gaza_week.focal_events.sum())),
 ("District-week focal total",10648,int(district_week.focal_events.sum())),
 ("Incomplete Gaza weeks",1,int((~gaza_week.complete_week).sum())),
 ("Precision-1 land focal",7576,int(incidents.geo_precision_1.sum())),
],columns=["Check","Phase 2 reference","Loaded value"])
checks["Match"]=[a==b for a,b in zip(checks["Phase 2 reference"],checks["Loaded value"])]
display(checks)
print("All checks match Phase 2." if checks.Match.all() else "WARNING: reconcile the mismatched inputs before interpreting results.")
print("Incomplete week starts:",gaza_week.loc[~gaza_week.complete_week,"week_start"].dt.date.tolist())

# Task 1 — Univariate analysis
## Distributions, summary statistics, and frequency counts

**Unit:** one Gaza event in `master`, except where the weekly or spatial table is named. Describe variables on their own before comparing groups. Phase 2 retained meaningful structural blanks; do not treat every missing value as a data error.

### 1.1 Variable profile and numeric summary

In [ ]:
profile = pd.DataFrame({"dtype":master.dtypes.astype(str),"missing":master.isna().sum(),
 "missing_pct":(master.isna().mean()*100).round(2),"distinct_nonmissing":master.nunique(dropna=True)})
display(profile.sort_values(["missing_pct","distinct_nonmissing"],ascending=[False,False]))
print("Numeric summary")
display(master.select_dtypes(include="number").describe(percentiles=[.25,.5,.75,.9,.95,.99]).T)

**Interpretation:** [Summarize meaningful missingness, long-tailed numeric fields, and dominant categories. Use Phase 2's missingness decisions to explain structural blanks.]

### 1.2 Frequency counts for core categorical variables

In [ ]:
for col in ["period","event_type","sub_event_type","disorder_type","civilian_targeting",
            "targeting_flagged","political_violence","focal_event","geo_precision","time_precision","admin2"]:
    print(f"\n{col} — count and share")
    t=master[col].value_counts(dropna=False).rename("count").to_frame()
    t["share"]=t["count"]/len(master)
    display(t)
for col in ["actor1","actor2"]:
    print(f"\n{col} — top 15 including missing")
    t=master[col].value_counts(dropna=False).head(15).rename("count").to_frame()
    t["share_of_all_events"]=t["count"]/len(master)
    display(t)

### V1 — How are reported fatalities distributed?

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
axes[0].hist(master.fatalities.dropna(),bins=35,color=C_OTHER,edgecolor="white")
axes[0].set_title("How are reported fatalities distributed on the original scale?")
axes[0].set_xlabel("Reported fatalities per event"); axes[0].set_ylabel("Event records")
axes[1].hist(master.fatalities_log1p.dropna(),bins=35,color=C_FOCAL,edgecolor="white")
axes[1].set_title("How does log1p make the long tail easier to inspect?")
axes[1].set_xlabel("log1p(reported fatalities)"); axes[1].set_ylabel("Event records")
plt.tight_layout(); plt.show()
display(master.fatalities.describe(percentiles=[.5,.75,.9,.95,.99]).to_frame("fatalities"))

**Interpretation (1-2 sentences):** [Describe zero mass, typical values, and tail. Explain why the log view helps and why the original values remain important.]

### V2 — Which event types occur most often?

In [ ]:
counts=master.event_type.value_counts(dropna=False).sort_values()
ax=counts.plot(kind="barh",figsize=(8,4.5),color=C_OTHER)
ax.set_title("Which recorded event types appear most often in Gaza?")
ax.set_xlabel("Event records"); ax.set_ylabel("ACLED event type")
plt.tight_layout(); plt.show()

**Interpretation (1-2 sentences):** [Name the dominant categories. Clarify that volume is not the same as focal-event prevalence.]

### V3 — How are geographic precision codes distributed?

In [ ]:
prec=master.geo_precision.value_counts(dropna=False).sort_index()
ax=prec.plot(kind="bar",figsize=(6.5,4),color=C_WARN)
ax.set_title("How many Gaza events use each geographic precision code?")
ax.set_xlabel("Geographic precision code (1 is most precise)"); ax.set_ylabel("Event records")
plt.tight_layout(); plt.show()

**Interpretation (1-2 sentences):** [Describe the precision mix and its effect on location-level conclusions.]

# Task 2 — Bivariate and multivariate analysis
## Correlations, group comparisons, cross-tabs, and pairwise relationships

The completed contribution below compares periods, event types, focal status, severity, precision, actors, districts, and weeks. Counts and rates are presented together where group sizes differ. Each analysis keeps its unit explicit.

### Event-level bivariate comparisons

Source table for this section: `master` (political-violence records unless stated). Every share names its numerator and denominator.

### 3.1 Focal share before vs. after October 7, 2023 (BQ1)

**Question.** Does the focal share of political-violence records differ between the two periods? The periods are 644 and 725 days long, and the earlier one is sparse, so the chart pairs the share with per-week rates.

In [ ]:
t1 = (political.groupby("period")
      .agg(political_records=("event_id_cnty", "size"), focal=("focal_event", "sum"))
      .loc[[BEFORE, AFTER]])
t1["not_focal"] = t1.political_records - t1.focal
t1["focal_share"] = t1.focal / t1.political_records
t1["calendar_days"] = [PERIOD_DAYS[p] for p in t1.index]
t1["focal_per_week"] = t1.focal * 7 / t1.calendar_days
t1["political_per_week"] = t1.political_records * 7 / t1.calendar_days
display(t1.round({"focal_share": 3, "focal_per_week": 2, "political_per_week": 2}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# Left: focal share with numerator/denominator written on each bar
ax = axes[0]
bars = ax.bar(t1.index, t1.focal_share, color=[C_GREY, C_FOCAL], width=0.55)
for b, (_, r) in zip(bars, t1.iterrows()):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.012,
            f"{r.focal:,.0f} / {r.political_records:,.0f}\n{r.focal_share:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 0.62)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel("Focal events / political-violence records")
ax.set_xlabel("Period")
ax.set_title("Did the focal share of political-violence records\ndiffer before vs. after Oct 7, 2023?")

# Right: per-week rates (log scale because the two periods differ by about 150x)
ax = axes[1]
x = np.arange(len(t1)); w = 0.36
b1 = ax.bar(x - w / 2, t1.political_per_week, w, color=C_OTHER, label="Political-violence records")
b2 = ax.bar(x + w / 2, t1.focal_per_week, w, color=C_FOCAL, label="Focal events")
for bars_, vals in ((b1, t1.political_per_week), (b2, t1.focal_per_week)):
    for b, v in zip(bars_, vals):
        ax.text(b.get_x() + b.get_width() / 2, v * 1.12, f"{v:,.1f}", ha="center", va="bottom", fontsize=9)
ax.set_yscale("log"); ax.set_ylim(0.3, 700)
ax.set_xticks(x)
ax.set_xticklabels([f"{p}\n({PERIOD_DAYS[p]} days)" for p in t1.index])
ax.set_xlabel("Period (calendar days)")
ax.set_ylabel("Records per week (log scale)")
ax.set_title("How many records per week does each period\nhold, once period length is accounted for?")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation.** The focal share is 16.2% (63 of 389 records) before Oct 7, 2023 and 47.6% (10,651 of 22,364) from Oct 7 onward, and the weekly rate of focal events goes from about 0.7 to about 102.8 across periods of 644 and 725 days (BQ1). The earlier period rests on only 63 focal events, so its share is imprecise, and a difference in share alone cannot separate a change in events from a change in coding or reporting coverage.

### 3.2 Focal prevalence within each event type (BQ2)

**Question.** Within each recorded event type, what share of political-violence records meet the focal rule? Counts alone would mostly show which types are large, so each bar uses its own type's denominator. Groups with fewer than 30 records are hatched.

In [ ]:
t2 = (political.groupby("event_type")
      .agg(n=("event_id_cnty", "size"), focal=("focal_event", "sum")))
t2["focal_share"] = t2.focal / t2.n
t2 = t2.sort_values("focal_share")
overall_share = political.focal_event.mean()
display(t2.round({"focal_share": 4}))

fig, ax = plt.subplots(figsize=(10.5, 4.6))
small = (t2.n < 30).to_numpy()
bars = ax.barh(t2.index, t2.focal_share, color=[C_GREY if s else C_FOCAL for s in small])
for b, s in zip(bars, small):
    if s:
        b.set_hatch("//")
for y, (_, r) in enumerate(t2.iterrows()):
    note = "  (n < 30)" if r.n < 30 else ""
    ax.text(r.focal_share + 0.012, y, f"{r.focal:,.0f} / {r.n:,.0f} = {r.focal_share:.1%}{note}", va="center", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
ax.axvline(overall_share, color="black", ls="--", lw=1)
ax.text(overall_share + 0.008, 2.5, f"all political-violence\nrecords: {overall_share:.1%}", fontsize=8.5, va="center",
        bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
ax.grid(axis="x")
ax.set_xlim(0, 1.5); ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Focal events / political-violence records of that event type")
ax.set_ylabel("Event type")
ax.set_title("Within each event type, what share of political-violence records is focal?")
plt.tight_layout()
plt.show()

**Interpretation.** Violence against civilians is almost entirely focal (808 of 814, 99.3%), Explosions/Remote violence is split (9,906 of 20,276, 48.9%), Battles have none (0 of 1,648), and Protests and Riots (7 and 8 records) have none (BQ2). The focal rule requires a civilian actor label, so part of this pattern is built into the coding; it describes how focal events are recorded, not which event types produce them, and the two small types cannot be judged.

### 3.3 Fatalities by focal status (BQ2)

**Question.** Do focal records report more fatalities than other political-violence records? Fatalities are heavily skewed with many zeros, so we compare full distributions (ECDF on a symmetric-log axis) and report median, IQR and percentiles instead of means alone. Fatalities are reported event totals, not civilian-death counts.

In [ ]:
def fatality_summary(df, by):
    g = df.groupby(by, observed=True).fatalities
    return pd.DataFrame({"n": g.size(), "share_zero": g.apply(lambda s: (s == 0).mean()),
                         "median": g.median(), "q1": g.quantile(0.25), "q3": g.quantile(0.75),
                         "p90": g.quantile(0.90), "mean": g.mean(), "max": g.max()})

print("Political-violence records, by focal status")
display(fatality_summary(political, "focal_event").round(2))
print("Same comparison split by period")
display(fatality_summary(political, ["period", "focal_event"]).round(2))

fig, ax = plt.subplots(figsize=(9.5, 5))
for flag, label, col in [(True, "Focal", C_FOCAL), (False, "Not focal", C_OTHER)]:
    v = np.sort(political.loc[political.focal_event == flag, "fatalities"].to_numpy())
    ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", color=col, lw=2, label=f"{label} (n = {len(v):,})")
ax.set_xscale("symlog", linthresh=1)
ticks = [0, 1, 2, 5, 10, 25, 50, 100, 250, 500]
ax.set_xticks(ticks); ax.set_xticklabels([str(t) for t in ticks]); ax.minorticks_off()
ax.set_xlim(0, 600); ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Reported fatalities per event (symmetric-log scale)")
ax.set_ylabel("Share of records at or below this value (ECDF)")
ax.set_title("Do focal records report more fatalities than other\npolitical-violence records?")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Interpretation.** Focal records report fatalities more often and in larger amounts: 31% report zero fatalities against 86% of other political-violence records, and the median is 2 against 0 (90th percentile 12 against 2). The gap is clear from Oct 7 onward (focal median 2, non-focal 0), while before Oct 7 both medians are 0 and only 63 records are focal (BQ2); fatalities are reported event totals, not confirmed civilian deaths, and a zero can mean that no fatality figure was recorded.

### 3.4 Fatalities by event type and focal status (BQ2)

**Question.** Because more than half of all records report zero fatalities, one number cannot describe severity. The left panel shows how often a group reports at least one fatality; the right panel shows how large the reported totals are among records that report at least one. Groups with fewer than 30 records are left out of the chart and kept in the table.

In [ ]:
short = {"Explosions/Remote violence": "Explosions", "Violence against civilians": "Violence vs. civilians"}
political["group"] = (political.event_type.map(lambda t: short.get(t, t))
                      + np.where(political.focal_event, " · focal", " · not focal"))

g5 = political.groupby("group").fatalities
t5 = pd.DataFrame({"n": g5.size(), "share_with_fatality": g5.apply(lambda s: (s > 0).mean()),
                   "events_with_fatality": g5.apply(lambda s: (s > 0).sum()),
                   "median_if_any": g5.apply(lambda s: s[s > 0].median()),
                   "q1_if_any": g5.apply(lambda s: s[s > 0].quantile(0.25)),
                   "q3_if_any": g5.apply(lambda s: s[s > 0].quantile(0.75)),
                   "max": g5.max()}).sort_values("n", ascending=False)
t5["shown_in_chart"] = t5.n >= 30
display(t5.round(3))

order = [g for g in t5.index[t5.n >= 30]]
palette5 = {g: (C_FOCAL if g.endswith("· focal") else C_OTHER) for g in order}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

ax = axes[0]
vals = t5.loc[order, "share_with_fatality"]
bars = ax.barh(order, vals, color=[palette5[g] for g in order])
for y, g in enumerate(order):
    r = t5.loc[g]
    ax.text(r.share_with_fatality + 0.01, y, f"{r.events_with_fatality:,.0f} / {r.n:,.0f} = {r.share_with_fatality:.0%}", va="center", fontsize=9)
ax.invert_yaxis(); ax.set_xlim(0, 1.0); ax.grid(axis="x")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Records with at least 1 reported fatality / all records in group")
ax.set_ylabel("Event type · focal status")
ax.set_title("How often does each group report any fatality?")

ax = axes[1]
sub = political[political.group.isin(order) & (political.fatalities > 0)]
sns.boxplot(data=sub, x="fatalities", y="group", hue="group", order=order, hue_order=order,
            palette=palette5, legend=False, orient="h", fliersize=2, linewidth=1, ax=ax)
ax.set_xscale("log")
ax.set_yticks(range(len(order)))
ax.set_yticklabels([f"{g}\n(n = {int((sub.group == g).sum()):,})" for g in order], fontsize=9)
ax.set_xlabel("Reported fatalities per event (log scale), records with at least 1")
ax.set_ylabel("")
ax.set_title("Among records reporting a fatality, how large\nare the reported totals?")
plt.tight_layout()
plt.show()

**Interpretation.** Within Explosions, 69% of focal records (6,869 of 9,906) report at least one fatality against 8% of non-focal ones (874 of 10,370), yet among records that do report a fatality the typical total is the same (median 3 in both); Battles have the largest typical totals (median 10, n = 760) and Violence against civilians the smallest (median 2, n = 519) (BQ2). So the focal group differs mainly in how often a fatality is reported rather than in how large it is, and every comparison depends on whether a fatality was recorded at all.

### 3.5 Location precision and focal share, stratified by period (BQ5)

**Question.** Does the focal share depend on how precisely ACLED coded the location? Precision (1 = most precise) is a quality category, not a distance. A precision difference can reflect documentation practices or coverage rather than a real difference in events, and the earlier period has very few records, so the comparison is made *within* each period. Bars with fewer than 30 records are hatched.

In [ ]:
t6 = (political.groupby(["period", "geo_precision"], observed=True)
      .agg(n=("event_id_cnty", "size"), focal=("focal_event", "sum")))
t6["focal_share"] = t6.focal / t6.n
display(t6.round({"focal_share": 3}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, period, col in zip(axes, [BEFORE, AFTER], [C_GREY, C_FOCAL]):
    d = t6.loc[period].reindex([1, 2, 3])
    bars = ax.bar([f"Precision {k}" for k in d.index], d.focal_share, color=col, width=0.6)
    for b, (_, r) in zip(bars, d.iterrows()):
        if r.n < 30:
            b.set_hatch("//"); b.set_alpha(0.55)
        ax.text(b.get_x() + b.get_width() / 2, r.focal_share + 0.015,
                f"{r.focal:,.0f} / {r.n:,.0f}\n{r.focal_share:.1%}", ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, 0.85)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Location precision code (1 = most precise)")
    ax.set_title(f"{period}\n(n = {int(d.n.sum()):,} political-violence records)")
axes[0].set_ylabel("Focal events / political-violence records")
fig.suptitle("Is the focal share different for records with less precise locations, within each period?",
             fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Robustness: does the precision 1 vs 2 gap survive inside each district and inside Explosions? (Oct 7 onward)
post_pol = political[political.period == AFTER]
by_district = post_pol[post_pol.geo_precision.isin([1, 2])].groupby(["admin2", "geo_precision"], observed=True).focal_event.agg(["size", "mean"]).unstack()
by_district.columns = [f"{'n' if a == 'size' else 'focal share'}, precision {b}" for a, b in by_district.columns]
print("Oct 7 onward: focal share by district and precision (precision 3 omitted, n < 70 in every district)")
display(by_district.round(3))
expl = post_pol[post_pol.event_type == "Explosions/Remote violence"]
print("Oct 7 onward, Explosions/Remote violence only")
display(expl.groupby("geo_precision", observed=True).focal_event.agg(n="size", focal="sum", focal_share="mean").round(3))

**Interpretation.** From Oct 7 onward the focal share is 59.6% for precision 1 (7,577 of 12,704) and 31.7% for precision 2 (3,040 of 9,583); before Oct 7 the same ordering appears at lower levels (21.7% vs. 14.2%), and precision 3 has too few records to read (8 and 77) (BQ5). The precision 1 vs. 2 gap holds inside all five districts (for example Deir El Balah 67.2% vs. 31.5%) and within Explosions (62.7% vs. 31.3%), so neither district nor event type explains it alone; these fields cannot tell us whether it reflects documentation practice, coverage or something else, which makes the precision-1 sensitivity run in Phase 4 important.

### 3.6 Recorded actor labels and sub-event types among focal events (BQ2)

**Question.** Which recorded `actor1` labels appear in focal events, and with which sub-event types? The focal rule requires "Civilians (Palestine)" in one actor field, so we first check where that label sits. Labels describe ACLED coding and are not verified roles. The five most frequent `actor1` labels are shown individually and the rest are grouped, with the number of grouped labels stated. Original labels are not merged.

In [ ]:
focal = master.loc[master.focal_event].copy()
print("Focal events:", len(focal))
print("actor2 == 'Civilians (Palestine)':", int((focal.actor2 == "Civilians (Palestine)").sum()),
      "| actor2 blank:", int(focal.actor2.isna().sum()),
      "| actor1 == 'Civilians (Palestine)':", int((focal.actor1 == "Civilians (Palestine)").sum()))
print("Distinct actor1 labels among focal events:", focal.actor1.nunique())
print("\nEvent type vs. sub-event type among focal events")
display(pd.crosstab(focal.sub_event_type, focal.event_type))

ct = pd.crosstab(focal.actor1, focal.sub_event_type)
totals = ct.sum(axis=1).sort_values(ascending=False)
TOP_N = 5
keep = totals.index[:TOP_N]
ct_show = ct.loc[keep].copy()
ct_show.loc[f"All other actor1 labels ({len(totals) - TOP_N} labels)"] = ct.loc[~ct.index.isin(keep)].sum()
ct_show = ct_show[ct_show.sum().sort_values(ascending=False).index]
ct_show.index = [f"{i}  (n = {n:,})" for i, n in ct_show.sum(axis=1).items()]
display(ct_show)

fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(ct_show.where(ct_show > 0), annot=ct_show.astype(int).astype(str), fmt="",
            cmap="Reds", norm=LogNorm(vmin=1, vmax=ct_show.to_numpy().max()),
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Focal events (log colour scale)"}, ax=ax)
ax.grid(False)
ax.set_xlabel("Sub-event type (blank cell = 0 events)")
ax.set_ylabel("Recorded actor1 label")
ax.set_title("Which recorded actor1 labels appear in focal events, and under which sub-event types?")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.show()

**Interpretation.** All 10,714 focal events carry "Civilians (Palestine)" in `actor2` and none in `actor1`, so the actor view rests on `actor1`, where "Military Forces of Israel (2022-)" appears in 10,493 events (97.9%), mostly under Air/drone strike (8,392) and Shelling/artillery/missile attack (1,436); the Attack sub-event type (792 events) is where the other labels appear, such as Sahm Unit (90) and Hamas Movement (35) (BQ2). These are ACLED's recorded labels, not verified roles, and one label dominates so heavily that the other rows are hard to compare, so any statement about who did what needs the event notes.

### Grouped, multivariate, and temporal comparisons

The unit of analysis changes in this section. Chart 7 uses event records, Charts 8 and 10 use district-weeks, and Charts 9 and 11 use Gaza-wide weeks. The district-week table leaves out the 4 events with no district (3 of them focal), so its focal total (10,648) is 3 below the Gaza-wide total (10,651).

### 4.1 Event type, period and focal status together (BQ1, BQ2)

**Question.** Did the mix of recorded event types change between the periods, and did the focal share *within* a type change? The left panel shows composition (each bar sums to 100% of that period's political-violence records). The right panel shows the focal share for the only two event types that contain focal events. Counts for every type, including the very small ones, are in the table.

In [ ]:
type_order = ["Explosions/Remote violence", "Battles", "Violence against civilians", "Protests", "Riots"]
t7 = (political.groupby(["period", "event_type"])
      .agg(records=("event_id_cnty", "size"), focal=("focal_event", "sum")))
t7["focal_share"] = t7.focal / t7.records
display(t7.round({"focal_share": 3}))

comp = (pd.crosstab(political.period, political.event_type)
        .reindex(index=[BEFORE, AFTER], columns=type_order, fill_value=0))
comp_share = comp.div(comp.sum(axis=1), axis=0)
type_colors = dict(zip(type_order, ["#315A7D", "#9AA5B1", "#8D3B42", "#E0A458", "#6B8F71"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.3, 1]})
ax = axes[0]
for i, period in enumerate([BEFORE, AFTER]):
    left = 0
    for t in type_order:
        wdt = comp_share.loc[period, t]
        ax.barh(i, wdt, left=left, color=type_colors[t], edgecolor="white", label=t if i == 0 else None)
        if wdt >= 0.03:
            ax.text(left + wdt / 2, i, f"{wdt:.0%}", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
        left += wdt
ax.set_yticks([0, 1])
ax.set_yticklabels([f"{p}\n(n = {int(comp.loc[p].sum()):,})" for p in [BEFORE, AFTER]])
ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Share of the period's political-violence records")
ax.set_ylabel("Period")
ax.set_title("Did the event-type mix of political-violence\nrecords change between periods?")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)

ax = axes[1]
two = ["Explosions/Remote violence", "Violence against civilians"]
xs = np.arange(len(two)); w = 0.36
for j, (period, col) in enumerate([(BEFORE, C_GREY), (AFTER, C_FOCAL)]):
    vals = [t7.loc[(period, t), "focal_share"] for t in two]
    bars = ax.bar(xs + (j - 0.5) * w, vals, w, color=col, label=period)
    for b, t in zip(bars, two):
        r = t7.loc[(period, t)]
        ax.text(b.get_x() + b.get_width() / 2, r.focal_share + 0.015, f"{r.focal:,.0f}/{r.records:,.0f}\n{r.focal_share:.0%}",
                ha="center", va="bottom", fontsize=8.5)
ax.set_xticks(xs); ax.set_xticklabels(["Explosions/\nRemote violence", "Violence against\ncivilians"])
ax.set_ylim(0, 1.2); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Event type")
ax.set_ylabel("Focal events / political-violence records of that type")
ax.set_title("Within these two types, did the focal share\nchange between periods?")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation.** Explosions/Remote violence grows from 77% (298 of 389) to 89% (19,978 of 22,364) of political-violence records while Battles fall from 14% to 7%; within types, the focal share of Explosions rises from 13.1% (39 of 298) to 49.4% (9,867 of 19,978), while Violence against civilians stays near 100% in both periods (24 of 24, then 784 of 790) (BQ1, BQ2). The rise in the overall focal share is therefore carried by Explosions/Remote violence, but its earlier base of 298 records is small and Protests and Riots are too rare (12 records before, 3 after) to show.

### Actor and event-profile cross-tab

The actor view is already covered by Chart 6, so we do not repeat it here.

### 4.3 District by week: where and when are focal counts high? (BQ4)

**Question.** Which districts have persistent or sudden high-count weeks? Every district has a row for every week; weeks with no recorded focal events stay in the grid with value 0. The final partial week is greyed out and left out of the colour scale. This chart shows counts, which reflect volume; Chart 10 shows shares.

In [ ]:
dw = district_week.copy()
dist_order = dw.groupby("admin2").focal_events.sum().sort_values(ascending=False).index
heat = dw.pivot(index="admin2", columns="week_start", values="focal_events").loc[dist_order].astype(float)
complete = dw.drop_duplicates("week_start").set_index("week_start").complete_week.reindex(heat.columns).to_numpy()
heat_plot = heat.copy()
heat_plot.loc[:, ~complete] = np.nan  # keep the partial week out of the colour scale

dist_summary = dw.groupby("admin2").agg(focal=("focal_events", "sum"), political=("political_events", "sum"),
                                         median_weekly_focal=("focal_events", "median"),
                                         max_weekly_focal=("focal_events", "max"),
                                         weeks_without_political=("political_events", lambda s: int((s == 0).sum()))).loc[dist_order]
dist_summary["focal_share"] = dist_summary.focal / dist_summary.political
display(dist_summary.round({"focal_share": 3}))
print("Highest district-weeks by focal count")
display(dw.nlargest(6, "focal_events")[["admin2", "week_start", "focal_events", "political_events", "focal_share"]].round({"focal_share": 3}))

fig, ax = plt.subplots(figsize=(15, 3.8))
sns.heatmap(heat_plot, cmap="rocket_r", xticklabels=False, cbar_kws={"label": "Recorded focal events per week", "pad": 0.015}, ax=ax)
for j in np.where(~complete)[0]:
    ax.add_patch(Rectangle((j, 0), 1, len(heat), facecolor="#D9D9D9", edgecolor="none", hatch="///"))
    ax.text(j + 0.5, -0.15, "partial week", ha="right", va="bottom", fontsize=8)
first_of_quarter = ~pd.Series(pd.PeriodIndex(heat.columns, freq="Q")).duplicated().to_numpy()
pos = np.where(first_of_quarter)[0]
ax.set_xticks(pos + 0.5)
ax.set_xticklabels(heat.columns[pos].strftime("%b %Y"), rotation=0, fontsize=9)
ax.grid(False)
ax.set_xlabel("Week start (Saturday), labelled at the first week of each quarter")
ax.set_ylabel("District (admin2)")
ax.set_title("Which Gaza districts have high focal-event weeks, and when?")
plt.tight_layout()
plt.show()

**Interpretation.** Gaza City holds the six highest district-weeks (102 focal events in the week of Nov 4, 2023, and 77 to 84 in each of three weeks in September 2025), and every district turns pale in the same stretch of early 2025 (BQ4). Counts reflect volume as well as focus (Gaza City records 6,903 political-violence records against 2,921 in Rafah), the final partial week is excluded, and a zero cell means zero recorded events, so Chart 10 is needed to read the share.

### Gaza-wide weekly counts and share (BQ4)

**Question.** How do weekly focal counts and the focal share move over time, and are the two moving together? The top panel shows counts, the bottom panel shows the share, on the same week axis. The hollow marker is the incomplete final week. The shaded band marks the five weeks with the lowest focal counts; we describe them and do not explain them, because the files hold no source for the cause.

In [ ]:
gw = gaza_week.sort_values("week_start").copy()
low5 = gw.nsmallest(5, "focal_events")
low_span = (low5.week_start.min(), low5.week_end.max())
partial = gw[~gw.complete_week]
full = gw[gw.complete_week]
post_share = master.loc[master.period == AFTER, "focal_event"].sum() / political.loc[political.period == AFTER].shape[0]
print("Five lowest focal-count weeks:", [d.date().isoformat() for d in sorted(low5.week_start)])
display(low5.sort_values("week_start")[["week_start", "recorded_events", "political_events", "focal_events", "focal_share"]].round({"focal_share": 3}))
print("Mean weekly focal events by quarter (complete weeks)")
display(full.groupby(full.week_start.dt.to_period("Q"))[["focal_events", "political_events"]].mean().round(1))

fig, axes = plt.subplots(2, 1, figsize=(13, 7.2), sharex=True, gridspec_kw={"height_ratios": [1.2, 1]})
ax = axes[0]
ax.plot(full.week_start, full.political_events, color=C_OTHER, lw=1.8, label="Political-violence records")
ax.plot(full.week_start, full.focal_events, color=C_FOCAL, lw=1.8, label="Focal events")
ax.scatter(partial.week_start, partial.focal_events, facecolor="white", edgecolor=C_FOCAL, zorder=5, label="Incomplete final week (focal)")
ax.scatter(partial.week_start, partial.political_events, facecolor="white", edgecolor=C_OTHER, zorder=5)
ax.axvspan(*low_span, color="#E0A458", alpha=0.25, label="Five lowest focal-count weeks")
ax.set_ylabel("Records per week")
ax.set_title("How do weekly political-violence and focal counts change across Gaza?")
ax.legend(loc="upper right", ncol=2)

ax = axes[1]
ax.plot(full.week_start, full.focal_share, color=C_OTHER, lw=1.8, label="Weekly focal share")
ax.scatter(partial.week_start, partial.focal_share, facecolor="white", edgecolor=C_OTHER, zorder=5, label="Incomplete final week")
ax.axhline(post_share, color="black", ls="--", lw=1, label=f"Pooled share, Oct 7 onward ({post_share:.1%})")
ax.axvspan(*low_span, color="#E0A458", alpha=0.25)
ax.set_ylim(0, 0.85)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel("Focal events / political-violence records")
ax.set_xlabel("Week start (Saturday)")
ax.set_title("What share of each week's political-violence records is focal?")
ax.legend(loc="upper right", ncol=3)
plt.tight_layout()
plt.show()

**Interpretation.** Weekly focal events peak at 254 in the week of Oct 28, 2023, run between 61 and 120 through 2024, fall to 15 to 23 in the five lowest weeks (week starts Jan 25 to Feb 22, 2025), and average about 125 per week in Q2 and Q3 2025; the focal share ranges from 30% to 73% and sits above the pooled 47.6% in 17% of 2024 weeks but 89% of 2025 weeks (BQ4). The share does not fall with the counts in the low weeks (38% to 70%), so counts and shares must be read separately, and with only 22 to 40 political-violence records in those weeks the share is volatile; the cause of the dip cannot be read from these files.

### District focal share with denominators (BQ4)

**Question.** After accounting for how much political violence each district records, do districts differ in focal share, and is the share stable week to week? Each panel shows the weekly share (thin line, gaps where the denominator is zero), a 4-week pooled share (focal events over political-violence records in the same four weeks), and the district's overall share (dashed). The incomplete final week is left out. The panel titles and table carry the denominators.

In [ ]:
dwc = district_week[district_week.complete_week].sort_values(["admin2", "week_start"]).copy()
roll = lambda col: dwc.groupby("admin2")[col].transform(lambda s: s.rolling(4, min_periods=4).sum())
dwc["focal_4w"], dwc["political_4w"] = roll("focal_events"), roll("political_events")
dwc["share_4w"] = dwc.focal_4w / dwc.political_4w.where(dwc.political_4w > 0)

tbl = (dwc.groupby("admin2").agg(focal=("focal_events", "sum"), political=("political_events", "sum"),
                                  undefined_weeks=("focal_share", lambda s: int(s.isna().sum()))).loc[dist_order])
tbl["overall_share"] = tbl.focal / tbl.political
display(tbl.round({"overall_share": 3}))

fig, axes = plt.subplots(3, 2, figsize=(13, 9.5), sharex=True, sharey=True)
axes = axes.ravel()
for ax, dname in zip(axes, dist_order):
    d = dwc[dwc.admin2 == dname]
    ax.plot(d.week_start, d.focal_share, color=C_GREY, lw=0.9, label="Weekly share")
    ax.plot(d.week_start, d.share_4w, color=C_FOCAL, lw=2, label="4-week pooled share")
    ax.axhline(tbl.loc[dname, "overall_share"], color="black", ls="--", lw=1, label="District overall share")
    ax.set_title(f"{dname}: {tbl.loc[dname, 'focal']:,.0f} focal / {tbl.loc[dname, 'political']:,.0f} political records")
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.tick_params(axis="x", labelrotation=30)
axes[0].legend(loc="lower left", fontsize=8)
axes[3].tick_params(axis="x", labelbottom=True, labelrotation=30)
for ax in axes[-3:-1]:
    ax.set_xlabel("Week start")
for ax in axes[::2][:3]:
    ax.set_ylabel("Focal share")
# Last panel: the denominators as a table
ax = axes[-1]; ax.axis("off")
cell = [[i, f"{r.focal:,.0f}", f"{r.political:,.0f}", f"{r.overall_share:.1%}", int(r.undefined_weeks)] for i, r in tbl.iterrows()]
t = ax.table(cellText=cell, colLabels=["District", "Focal", "Political", "Share", "Weeks w/o\npolitical"], loc="center", cellLoc="center")
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.6)
ax.set_title("Denominators (complete weeks only)")
fig.suptitle("Do districts differ in focal share, and how stable is the share from week to week?", fontsize=12, fontweight="bold", y=1.0)
plt.tight_layout()
plt.show()

**Interpretation.** Overall focal share runs from 40.4% in North Gaza (1,325 of 3,277) to 52.1% in Deir El Balah (2,257 of 4,335), with Gaza City at 50.0% (3,411 of 6,825); the 4-week share starts high in every district, falls below the district average for most of 2024 (Gaza City is above it in only 8% of 2024 weeks), and from June 2025 Gaza City and Rafah stay above it in every week (BQ4). Rafah swings most (20.0% in the week of Aug 17, 2024 to 87.8% on Jul 5, 2025), and weekly district shares rest on small denominators (North Gaza has 2 weeks with no political-violence records), so single-week spikes in share should not be over-read.

### Pairwise correlations and categorical associations (BQ4, BQ5)

**Question.** Which weekly measures move together, and which categorical fields are associated? Two different measures are needed because the variable types differ.

- **Left, Spearman rank correlation** between weekly Gaza-wide measures (complete weeks only). † marks pairs that are related by construction (a subset or a denominator relationship, or overlapping windows), so a high value there is expected and says little. ‡ marks a variable correlated with its own earlier values (serial dependence). The sample size is lower for the lag columns because early weeks have no eligible history.
- **Right, Cramér's V** between categorical fields of political-violence records (0 = no association, 1 = perfect). Sub-event type is nested inside event type, so that pair is 1.0 by definition. V can be inflated when many cells are sparse, so the share of sparse cells is shown below the chart.

Neither measure shows cause.

In [ ]:
# ---- Left: Spearman on weekly Gaza-wide numbers (complete weeks only) ----
wk_cols = ["recorded_events", "political_events", "focal_events", "focal_share", "previous_week_focal", "previous_four_weeks_focal"]
wk_lab = {"recorded_events": "Recorded events", "political_events": "Political-violence records", "focal_events": "Focal events",
          "focal_share": "Focal share", "previous_week_focal": "Focal events, previous week", "previous_four_weeks_focal": "Focal events, previous 4 weeks"}
wk = gaza_week.loc[gaza_week.complete_week, wk_cols]
rho = wk.corr(method="spearman")
n_pair = wk.notna().astype(int).T.dot(wk.notna().astype(int))
mechanical = [{"recorded_events", "political_events"}, {"political_events", "focal_events"},
              {"focal_events", "focal_share"}, {"political_events", "focal_share"},
              {"previous_week_focal", "previous_four_weeks_focal"}]
serial = [{"focal_events", "previous_week_focal"}, {"focal_events", "previous_four_weeks_focal"}]
def mark(a, b):
    s = {a, b}
    return "†" if s in mechanical else ("‡" if s in serial else "")
annot_r = pd.DataFrame([[f"{rho.loc[a, b]:.2f}{mark(a, b)}" if a != b else "1" for b in wk_cols] for a in wk_cols], index=wk_cols, columns=wk_cols)
print(f"Weekly Spearman: {len(wk)} complete weeks; pairwise n ranges from {int(n_pair.to_numpy().min())} to {int(n_pair.to_numpy().max())}")

# ---- Right: Cramér's V (plain, uncorrected) on political-violence records ----
def cramers_v(a, b):
    ok = a.notna() & b.notna()
    ct = pd.crosstab(a[ok].astype(str), b[ok].astype(str))
    if min(ct.shape) < 2:
        return np.nan, np.nan
    chi2, _, _, expected = chi2_contingency(ct, correction=False)
    return np.sqrt(chi2 / (ct.to_numpy().sum() * (min(ct.shape) - 1))), (expected < 5).mean()

cat_vars = {"Focal event": political.focal_event, "Event type": political.event_type, "Sub-event type": political.sub_event_type,
            "Period": political.period, "District": political.admin2, "Location precision": political.geo_precision,
            "Date precision": political.time_precision}
names = list(cat_vars)
V = pd.DataFrame(np.nan, index=names, columns=names); sparse = V.copy()
for i, a in enumerate(names):
    for b in names[: i + 1]:
        if a == b:
            V.loc[a, b] = 1.0
        else:
            V.loc[a, b], sparse.loc[a, b] = cramers_v(cat_vars[a], cat_vars[b])

fig, axes = plt.subplots(1, 2, figsize=(16, 6.2), gridspec_kw={"width_ratios": [1, 1.05]})
sns.heatmap(rho, mask=np.triu(np.ones_like(rho, dtype=bool), k=1), annot=annot_r, fmt="", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Spearman rho"}, ax=axes[0])
for a_ in axes:
    a_.grid(False)
axes[0].set_xticklabels([wk_lab[c] for c in wk_cols], rotation=35, ha="right")
axes[0].set_yticklabels([wk_lab[c] for c in wk_cols], rotation=0)
axes[0].set_title(f"Which weekly Gaza measures move together?\n(Spearman, {len(wk)} complete weeks; † built-in overlap, ‡ serial dependence)")
sns.heatmap(V, mask=np.triu(np.ones_like(V, dtype=bool), k=1), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Cramér's V"}, ax=axes[1])
axes[1].set_xticklabels(names, rotation=35, ha="right"); axes[1].set_yticklabels(names, rotation=0)
axes[1].set_title(f"Which categorical fields are associated?\n(Cramér's V, {len(political):,} political-violence records)")
plt.tight_layout()
plt.show()

print("Share of contingency-table cells with expected count < 5 (sparse cells inflate V)")
display((sparse * 100).round(0).fillna("").astype(str).replace({"nan": ""}))

**Interpretation.** Weekly focal events move with recorded events (rho = 0.60) and with their own previous week (0.70) and previous four weeks (0.66), which supports using prior-week history as a feature in Phase 4 (BQ4); among categorical fields, focal status is moderately associated with sub-event type (V = 0.56), event type (0.32) and location precision (0.28) but weakly with period and district (0.08 each). Pairs marked † are related by construction (for example 0.97 between recorded events and political-violence records) and say little, the period value is held down by the very unequal group sizes (389 vs. 22,364 records), V is inflated where cells are sparse (see the table), and none of this shows cause.

# Task 3 — Visualization requirement and chart register

The Phase 3 guide requires **at least eight purposeful visualizations**. This notebook contains well over that minimum. Figures remain adjacent to their analysis and interpretation; the register below makes the coverage easy to review without repeating plots.

| Chart family | Included visualizations | Located in |
|---|---|---|
| Univariate | V1 fatalities distribution; V2 event-type frequency; V3 precision frequency | Task 1 |
| Bivariate/multivariate | Period focal share/rate; event-type focal share; fatality ECDF; severity by type/status; precision by period; actor/sub-event cross-tab; event-type mix; district-week counts; weekly trends; district rolling shares; correlation/association | Task 2 |
| Hypothesis tests | Period test with intervals/within-type check; event-type test with effect and residual view | Task 4 |
| Pattern spotting | Imbalance; fatality tail/timing; unusual weeks; monthly trend; seasonality/weekday; autocorrelation; district profiles; descriptive clusters; spatial precision; coordinate concentration; DBSCAN sensitivity | Task 5 |

### Visual quality checklist
- [ ] Title states a question relevant to BQ1-BQ5.
- [ ] Axes carry units; legends explain colors and markers.
- [ ] One-to-two-sentence interpretation directly follows each figure.
- [ ] Counts, rates, and shares are not conflated; denominators are visible.
- [ ] Small groups, precision, incomplete weeks, and recording limitations are explained.

Remove figures that are decorative or repeat a comparison without adding information.

# Task 4 — Hypothesis testing
## Statistical tests, p-values, and plain-language conclusions

Both tests use political-violence records as the denominator. They include expected-count checks, an appropriate p-value procedure, effect sizes, and limitations. Because records may be related across time and place, interpret these as exploratory associations.

The helper functions below keep the two tests consistent. `focal_table` builds each contingency table, `preflight` prints the smallest expected count (the check that decides which p-value procedure is valid), `wilson_ci` gives 95% intervals for the charts, and `fmt_p` prints very small p-values readably.

In [ ]:
def focal_table(frame, by, order):
    """Focal / not-focal counts for each group in `order`. reindex keeps empty groups, and it reads
    [True, False] as column labels (a [True, False] list passed to .loc would be read as a boolean mask)."""
    tab = pd.crosstab(frame[by], frame.focal_event).reindex(index=order, columns=[True, False], fill_value=0)
    tab.columns = ["Focal", "Not focal"]
    return tab

def expected_counts(table):
    """Expected counts under independence: row total x column total / grand total."""
    t = np.asarray(table, dtype=float)
    return np.outer(t.sum(axis=1), t.sum(axis=0)) / t.sum()

def preflight(label, table):
    """Print the sparse-cell diagnostic that decides which p-value procedure is valid."""
    e = expected_counts(table)
    print(f"{label}: smallest expected count = {e.min():.2f}; cells below 5 = {(e < 5).sum()} of {e.size}")
    return e

def wilson_ci(k, n):
    """95% Wilson interval for a share; it stays inside 0 to 1 and works when k = 0 or k = n."""
    ci = binomtest(int(k), int(n)).proportion_ci(confidence_level=0.95, method="wilson")
    return ci.low, ci.high

def fmt_p(p, monte_carlo=False):
    """Readable p-value. Floating point underflows to 0.0 for very small chi-square p-values, and a
    Monte Carlo p-value cannot go below 1 / (N_RESAMPLES + 1)."""
    if monte_carlo:
        floor = 1 / (N_RESAMPLES + 1)
        return f"p < {floor:.5f}" if np.isclose(p, floor) else f"p = {p:.5f}"
    return "p < 1e-300" if p == 0 else f"p = {p:.2e}"

### Hypothesis A: focal share before vs. after October 7, 2023 (BQ1)

**Question.** Is the focal share of political-violence records different before and after October 7, 2023?

**H0:** the focal share is the same in both periods. <br> 
**H1:** it differs. <br>
**Test:** Pearson chi-square test of independence on the 2 × 2 table of period by focal status, without continuity correction because every expected count is large. The periods are 644 and 725 days long, but a share is a ratio inside each period, so period length does not enter the test; it matters only for the per-week rates reported in Phase 3. We also report Fisher's exact test and the continuity-corrected test to show that the p-value does not depend on that choice.

In [ ]:
period_table = focal_table(political, "period", [BEFORE, AFTER])
display(period_table.assign(Total=period_table.sum(axis=1),
                            Focal_share=(period_table.Focal / period_table.sum(axis=1)).round(3)))
preflight("A: period x focal", period_table)

tab_A = period_table.to_numpy()
chi2_A, p_A, dof_A, _ = chi2_contingency(tab_A, correction=False)   # Pearson chi-square, no continuity correction
p_A_yates = chi2_contingency(tab_A, correction=True)[1]             # sensitivity: with continuity correction
p_A_fisher = fisher_exact(tab_A)[1]                                 # sensitivity: exact test
n_A = tab_A.sum()
phi_A = np.sqrt(chi2_A / n_A)

# Effect sizes with 95% intervals (normal approximation; the ratios use the log scale)
(f_b, nf_b), (f_a, nf_a) = tab_A
n_b, n_a = f_b + nf_b, f_a + nf_a
share_b, share_a = f_b / n_b, f_a / n_a
diff = share_a - share_b
se_diff = np.sqrt(share_b * (1 - share_b) / n_b + share_a * (1 - share_a) / n_a)
rr = share_a / share_b
se_rr = np.sqrt((1 - share_a) / f_a + (1 - share_b) / f_b)
odds = (f_a * nf_b) / (f_b * nf_a)
se_or = np.sqrt(1 / f_a + 1 / nf_a + 1 / f_b + 1 / nf_b)

print(f"Pearson chi-square({dof_A}) = {chi2_A:,.2f}, {fmt_p(p_A)} (n = {n_A:,}, phi = {phi_A:.3f})")
print(f"Sensitivity: with continuity correction {fmt_p(p_A_yates)}; Fisher exact {fmt_p(p_A_fisher)}")
print(f"Focal share: {share_b:.1%} ({f_b:,} / {n_b:,}) before vs. {share_a:.1%} ({f_a:,} / {n_a:,}) from Oct 7 onward")
print(f"Difference = {diff * 100:+.1f} points (95% CI {(diff - Z * se_diff) * 100:+.1f} to {(diff + Z * se_diff) * 100:+.1f}); "
      f"risk ratio = {rr:.2f} ({rr * np.exp(-Z * se_rr):.2f} to {rr * np.exp(Z * se_rr):.2f}); "
      f"odds ratio = {odds:.2f} ({odds * np.exp(-Z * se_or):.2f} to {odds * np.exp(Z * se_or):.2f})")
decision_A = "Reject H0" if p_A < ALPHA / 2 else "Fail to reject H0"
print(f"Decision: {decision_A} (alpha = {ALPHA}, Bonferroni-adjusted to {ALPHA / 2} for the two primary tests)")

**Interpretation.** We reject H0 (chi-square(1) = 151.60, p = 7.8e-35; Fisher exact p = 3.8e-38). The focal share is 16.2% (63 of 389 records) before Oct 7, 2023 and 47.6% (10,651 of 22,364) from Oct 7 onward, a rise of 31.4 percentage points (95% CI 27.7 to 35.1). A record from Oct 7 onward is about 2.9 times as likely to be focal (risk ratio 2.94, 95% CI 2.34 to 3.69; odds ratio 4.71, 95% CI 3.59 to 6.17) (BQ1). In plain terms, a gap this large would almost never appear by chance if the focal share were the same in both periods. Phi is only 0.08 because the two groups are very unequal in size (389 vs. 22,364 records, as noted in Phase 3), so the share difference and the odds ratio describe the size of the change better. The earlier share rests on 63 focal events, so its interval is wide (12.9% to 20.2%), and the test cannot separate a real change in events from a change in coding or reporting coverage.

#### 4.3.1 Does the rise hold inside event types? (BQ1, BQ2)

**Question.** The overall rise could come from a change in the event-type mix instead of a change inside event types. In Phase 3 (Chart 7) Explosions/Remote violence grows from 77% to 89% of records, and Battles never contain focal events. We therefore repeat the comparison inside the two event types that contain focal events. This is a robustness check, not a third hypothesis. We use the chi-square test when every expected count is at least 5 and Fisher's exact test otherwise, and groups with fewer than 30 records are marked in the chart.

In [ ]:
# Within each event type that contains focal events: is the period difference still there?
within_tabs, within_rows = {}, []
for etype in EVENT_TYPES[:2]:
    t = focal_table(political.loc[political.event_type == etype], "period", [BEFORE, AFTER]).to_numpy()
    within_tabs[etype] = t
    e_min = expected_counts(t).min()
    if e_min >= 5:   # the large-sample test is valid only when every expected count is at least 5
        test_name, p_val = "Chi-square", chi2_contingency(t, correction=False)[1]
    else:
        test_name, p_val = "Fisher exact", fisher_exact(t)[1]
    within_rows.append({"Event type": etype,
                        "Before: focal / records": f"{t[0, 0]:,} / {t[0].sum():,}", "Before share": t[0, 0] / t[0].sum(),
                        "After: focal / records": f"{t[1, 0]:,} / {t[1].sum():,}", "After share": t[1, 0] / t[1].sum(),
                        "Smallest expected": e_min, "Test": test_name, "p-value": fmt_p(p_val)})
display(pd.DataFrame(within_rows).set_index("Event type").round({"Before share": 3, "After share": 3, "Smallest expected": 2}))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw={"width_ratios": [1, 1.3]})

# Left: the tested comparison, with 95% Wilson intervals
ax = axes[0]
shares = np.array([share_b, share_a])
cis = np.array([wilson_ci(f_b, n_b), wilson_ci(f_a, n_a)])
bars = ax.bar([BEFORE, AFTER], shares, color=[C_GREY, C_FOCAL], width=0.55,
              yerr=[shares - cis[:, 0], cis[:, 1] - shares], capsize=5, error_kw={"lw": 1.2})
for b, k, n, s, hi in zip(bars, (f_b, f_a), (n_b, n_a), shares, cis[:, 1]):
    ax.text(b.get_x() + b.get_width() / 2, hi + 0.015, f"{k:,} / {n:,}\n{s:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 0.7)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Period")
ax.set_ylabel("Focal events / political-violence records")
ax.set_title(f"Hypothesis A: did the focal share change?\n(chi-square {fmt_p(p_A)}; bars show 95% intervals)")

# Right: the same comparison inside the two event types that contain focal events
ax = axes[1]
xs, w = np.arange(2), 0.36
for j, (period, col) in enumerate([(BEFORE, C_GREY), (AFTER, C_FOCAL)]):
    vals, lows, highs, notes = [], [], [], []
    for etype in EVENT_TYPES[:2]:
        k, n = within_tabs[etype][j, 0], within_tabs[etype][j].sum()
        lo, hi = wilson_ci(k, n)
        vals.append(k / n); lows.append(k / n - lo); highs.append(hi - k / n)
        notes.append(f"{k:,}/{n:,}\n{k / n:.0%}" + ("\n(n < 30)" if n < 30 else ""))
    bars = ax.bar(xs + (j - 0.5) * w, vals, w, color=col, label=period, yerr=[lows, highs], capsize=4, error_kw={"lw": 1.1})
    for b, v, hi, note in zip(bars, vals, highs, notes):
        ax.text(b.get_x() + b.get_width() / 2, v + hi + 0.015, note, ha="center", va="bottom", fontsize=8.5)
ax.set_xticks(xs)
ax.set_xticklabels(["Explosions/\nRemote violence", "Violence against\ncivilians"])
ax.set_ylim(0, 1.3); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Event type")
ax.set_ylabel("Focal events / political-violence records of that type")
ax.set_title("Within these two event types, did the focal share change?")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation.** The rise holds inside Explosions/Remote violence: 13.1% (39 of 298 records) before vs. 49.4% (9,867 of 19,978) from Oct 7 onward (chi-square p = 1.5e-35). Inside Violence against civilians the share is already near 100% in both periods (24 of 24, then 784 of 790; Fisher exact p = 1.00, and the earlier group has fewer than 30 records), so there is no room for it to change (BQ1, BQ2). The overall rise is therefore not an effect of the changing event-type mix: it is carried by Explosions/Remote violence. The earlier Explosions share still rests on only 39 focal events.

### Hypothesis B: focal share across event types (BQ2)

**Question.** Does the focal share of political-violence records differ across the five recorded event types?

**H0:** the focal share is the same in every event type. <br>
**H1:** it differs for at least one event type. <br>
**Test:** chi-square test of independence on the 5 × 2 table of event type by focal status. The Phase 3 preflight found a smallest expected count of 3.30 and 4 of 10 cells below 5, because Protests (7 records) and Riots (8) are rare. The ordinary chi-square p-value is not reliable in that situation, so we estimate it by Monte Carlo resampling: 99,999 simulated tables with the same row and column totals and a fixed seed of 42, as in the Phase 2 missing-value test (Mugot, 2026). We keep all five event types in the table. Pearson residuals show which cells drive the statistic, and two sensitivity checks follow: pooling the two rare types so every expected count is at least 5, and comparing the two event types that contain focal events directly.

In [ ]:
type_table = focal_table(political, "event_type", EVENT_TYPES)
display(type_table.assign(Total=type_table.sum(axis=1),
                          Focal_share=(type_table.Focal / type_table.sum(axis=1)).round(3)))
exp_B = preflight("B: event type x focal", type_table)

# The preflight fails (see above), so the p-value comes from Monte Carlo resampling, as in Phase 2.
# All five event types stay in the table.
tab_B = type_table.to_numpy()
resampling = MonteCarloMethod(n_resamples=N_RESAMPLES, rng=np.random.default_rng(SEED))
res_B = chi2_contingency(tab_B, correction=False, method=resampling)
chi2_B, p_B = res_B.statistic, res_B.pvalue
dof_B = (tab_B.shape[0] - 1) * (tab_B.shape[1] - 1)   # the Monte Carlo result does not report degrees of freedom
n_B = tab_B.sum()
V_B = np.sqrt(chi2_B / (n_B * (min(tab_B.shape) - 1)))

print(f"Pearson chi-square({dof_B}) = {chi2_B:,.2f}, Monte Carlo {fmt_p(p_B, monte_carlo=True)} "
      f"({N_RESAMPLES:,} simulated tables with the same row and column totals)")
print(f"Cramér's V = {V_B:.3f} (n = {n_B:,})")
decision_B = "Reject H0" if p_B < ALPHA / 2 else "Fail to reject H0"
print(f"Decision: {decision_B} (alpha = {ALPHA}, Bonferroni-adjusted to {ALPHA / 2} for the two primary tests)")

# Which cells drive the statistic? Pearson residuals: (observed - expected) / sqrt(expected)
resid_B = pd.DataFrame((tab_B - exp_B) / np.sqrt(exp_B), index=EVENT_TYPES, columns=["Focal", "Not focal"])
print("\nPearson residuals (values beyond about +/-2 stand out; very large values mean far from independence)")
display(resid_B.round(1))

# Sensitivity 1: pool the two rare types so every expected count is at least 5, then use the ordinary test
pooled = type_table.copy()
pooled.loc["Protests and Riots (pooled)"] = pooled.loc[["Protests", "Riots"]].sum()
pooled = pooled.drop(["Protests", "Riots"])
preflight("B with Protests and Riots pooled (15 records)", pooled)
chi2_pool, p_pool, dof_pool, _ = chi2_contingency(pooled.to_numpy(), correction=False)
print(f"Pooled table: chi-square({dof_pool}) = {chi2_pool:,.2f}, {fmt_p(p_pool)}")

# Sensitivity 2: the only two event types that contain focal events, compared directly
pair = type_table.loc[EVENT_TYPES[:2]].to_numpy()
chi2_pair, p_pair, _, e_pair = chi2_contingency(pair, correction=False)
print(f"Violence against civilians vs. Explosions/Remote violence: chi-square(1) = {chi2_pair:,.2f}, {fmt_p(p_pair)} "
      f"(smallest expected count = {e_pair.min():.2f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.5, 1]})

# Left: focal share within each event type, with 95% Wilson intervals; groups with fewer than 30 records are hatched
ax = axes[0]
tb = type_table.assign(n=type_table.sum(axis=1))
tb["share"] = tb.Focal / tb.n
tb = tb.sort_values("share")
small = (tb.n < 30).to_numpy()
cis_B = np.array([wilson_ci(k, n) for k, n in zip(tb.Focal, tb.n)])
bars = ax.barh(tb.index, tb.share, color=[C_GREY if s else C_FOCAL for s in small],
               xerr=[tb.share - cis_B[:, 0], cis_B[:, 1] - tb.share], capsize=4, error_kw={"lw": 1.1})
for b, s in zip(bars, small):
    if s:
        b.set_hatch("//")
for y, (_, r) in enumerate(tb.iterrows()):
    note = "  (n < 30)" if r.n < 30 else ""
    ax.text(cis_B[y, 1] + 0.012, y, f"{r.Focal:,.0f} / {r.n:,.0f} = {r.share:.1%}{note}", va="center", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
overall = tab_B[:, 0].sum() / n_B
ax.axvline(overall, color="black", ls="--", lw=1)
ax.text(overall + 0.008, 0.5, f"all political-violence\nrecords: {overall:.1%}", fontsize=8.5, va="center",
        bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
ax.grid(axis="x")
ax.set_xlim(0, 1.5); ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Focal events / political-violence records of that event type")
ax.set_ylabel("Event type")
ax.set_title(f"Hypothesis B: does the focal share differ by event type?\n(Monte Carlo {fmt_p(p_B, monte_carlo=True)}; Cramér's V = {V_B:.2f})")

# Right: which cells push the table away from independence
ax = axes[1]
lim = float(np.abs(resid_B.to_numpy()).max())
sns.heatmap(resid_B, annot=True, fmt=".1f", cmap="RdBu_r", vmin=-lim, vmax=lim, center=0,
            linewidths=0.5, linecolor="white", cbar_kws={"label": "Pearson residual"}, ax=ax)
ax.grid(False)
ax.set_xlabel("Record status")
ax.set_ylabel("")
ax.set_title("Which cells differ most from what independence predicts?")
plt.tight_layout()
plt.show()

**Interpretation.** We reject H0 (chi-square(4) = 2,394.76, Monte Carlo p < 0.00001, Cramér's V = 0.32). None of the 99,999 simulated tables reached the observed statistic, so 0.00001 is the smallest p-value this run can report, not an exact value (BQ2). In plain terms, the focal share clearly depends on event type: Violence against civilians is almost entirely focal (808 of 814, 99.3%), Explosions/Remote violence is split (9,906 of 20,276, 48.9%) and Battles have none (0 of 1,648). The residuals put the largest departures from independence in Battles (focal cell -27.9), Violence against civilians (+21.7) and Explosions/Remote violence (+3.7), while Protests and Riots stay within about 2 of zero because 7 and 8 records cannot show a clear difference. The result does not depend on the sparse cells: with Protests and Riots pooled (15 records) every expected count is at least 5, the statistic is unchanged at 2,394.76 (both rare types have a focal share of 0) and the conclusion is the same, and Violence against civilians vs. Explosions/Remote violence alone gives chi-square(1) = 795.58 (p = 4.9e-175). The focal rule requires a civilian actor label, so part of this association is built into the coding: it describes how focal events are recorded across types, not which event types produce them.

### Summary of test results

The table below collects the two tests. It reads the values computed above, so it updates if the data are re-exported.

In [ ]:
summary = pd.DataFrame([
    {"Hypothesis": "A: focal share, before vs. after Oct 7, 2023", "BQ": "BQ1",
     "Test": "Pearson chi-square, 2 × 2", "Statistic": f"chi-square({dof_A}) = {chi2_A:,.2f}", "p-value": fmt_p(p_A),
     "Effect size": f"{diff * 100:+.1f} points; odds ratio {odds:.2f}; phi = {phi_A:.2f}", "Decision": decision_A},
    {"Hypothesis": "B: focal share across event types", "BQ": "BQ2",
     "Test": f"Chi-square, Monte Carlo p ({N_RESAMPLES:,} tables)", "Statistic": f"chi-square({dof_B}) = {chi2_B:,.2f}",
     "p-value": fmt_p(p_B, monte_carlo=True), "Effect size": f"Cramér's V = {V_B:.2f}", "Decision": decision_B},
]).set_index("Hypothesis")
print(f"Decision rule: reject H0 when p < {ALPHA / 2} (alpha = {ALPHA}, Bonferroni-adjusted for two primary tests)")
display(summary)

**Conclusions in plain language.**

- **Hypothesis A (BQ1): rejected H0.** If the focal share were the same before and after Oct 7, 2023, a gap as large as 16.2% vs. 47.6% would almost never occur by chance. From Oct 7 onward, a recorded political-violence event is about 2.9 times as likely to meet the focal rule, and the rise holds inside Explosions/Remote violence, so it is not only a change in the mix of event types.
- **Hypothesis B (BQ2): rejected H0.** The focal share depends strongly on event type: nearly all Violence against civilians records are focal, about half of Explosions/Remote violence records are, and none of the Battles records are. Protests and Riots are too rare to judge. Because the focal rule needs a civilian actor label, this pattern is partly a result of how the rule is built.

**What the tests do not show.**
- They do not show why the focal share changed. A change in events, in ACLED coding or in source coverage after Oct 7, 2023 would all produce the same table, and the earlier period holds only 389 political-violence records (63 focal).
- The p-values treat records as independent, but events cluster in time and place, so they are best read as strong exploratory evidence, not exact probabilities.
- Neither test shows intent, responsibility or a legal classification.

**For the insights writer and Phase 4 modelling.** (1) The rise in focal share is carried by Explosions/Remote violence (13.1% to 49.4%). (2) The event-type association is partly built into the focal rule and should be described that way. (3) The earlier period is sparse and its share is imprecise, which supports using it for description only and comparing weeks within the post-October 7 period in the models, as planned in Phase 2.

**References**

1. Group 7. *Phase 1: Dataset Selection and Business Problem* (business questions BQ1 to BQ5 and scope).
2. Group 7. *Phase 2: Data Preprocessing* (period definitions, focal-event rule, Monte Carlo chi-square test, `data/processed/` outputs).
3. Group 7. *Phase 3: Bivariate and Multivariate Analysis* (preflight diagnostics, Charts 1, 2, 7 and 11, handoff notes).
4. Mugot, C. J. S. *EDA Bivariate Analysis of Categorical Variables*, 2026.
5. SciPy documentation: `scipy.stats.chi2_contingency`, `MonteCarloMethod`, `fisher_exact`, `binomtest`.
6. ACLED. [Codebook](https://acleddata.com/methodology/acled-codebook). Definitions and interpretation limitations.
7. DS312. *Final Project Guide*, 1st Semester A.Y. 2026-2027 (Instructor: Nicole S. Menorias), hypothesis-testing requirement. Supplied course document.

# Task 5 — Pattern spotting
## Segments, trends, seasonality, suspicious values, and class imbalance

These analyses identify patterns and review candidates. An unusual value is not automatically invalid. Temporal changes may reflect reporting coverage as well as changes in recorded activity.

Two small helpers are used in Section 4 onward. `robust_z` is the *modified z-score*, which uses the median and the median absolute deviation (MAD) instead of the mean and standard deviation, so the very peaks we are looking for do not hide themselves. `trailing_robust_z` scores each week only against the **previous 8 weeks**, which is the baseline idea Phase 4 plans to use (trailing median/MAD). A score beyond ±3.5 is the usual rule of thumb for "unusual"; it is a screening rule, not a test.

In [ ]:
Z_CUT = 3.5   # conventional cut-off for the modified (robust) z-score

def robust_z(s):
    """Modified z-score: 0.6745 * (x - median) / MAD. Robust to the very peaks we are trying to find."""
    med = s.median()
    mad = (s - med).abs().median()
    return 0.6745 * (s - med) / mad

def trailing_robust_z(s, window=8, mad_floor=1.0):
    """Score each week against the previous `window` weeks only (median/MAD of its own recent history).
    The first `window` weeks have no score. mad_floor stops a flat history from dividing by zero."""
    vals = s.to_numpy(float)
    out = np.full(len(vals), np.nan)
    for i in range(window, len(vals)):
        hist = vals[i - window:i]
        med = np.median(hist)
        mad = max(np.median(np.abs(hist - med)), mad_floor)
        out[i] = 0.6745 * (vals[i] - med) / mad
    return pd.Series(out, index=s.index)

### Class imbalance

**Question.** Overall, focal events are 47.1% of political-violence records, which looks balanced. But Phase 4 will compare periods, event types and precision levels, and some of those slices are far from balanced. Where, and how thin is the smaller class? The table gives the focal share, the size of the smaller class and the majority-to-minority ratio for each slice. The weekly view at the bottom checks how often a week or a district-week is "empty".

*Source table: `master` (political-violence records) and the weekly tables. Unit: one record, one week, one district-week.*

In [ ]:
def imbalance_row(label, df):
    n = len(df); k = int(df.focal_event.sum()); minority = min(k, n - k)
    return {"Slice": label, "Records": n, "Focal": k, "Not focal": n - k,
            "Focal share": k / n, "Minority class": "focal" if k < n - k else "not focal",
            "Minority count": minority, "Majority : minority": (max(k, n - k) / minority) if minority else np.nan}

post = political[political.period == AFTER]
slices = [
    ("All political-violence records", political),
    ("Before Oct 7, 2023", political[political.period == BEFORE]),
    ("Oct 7, 2023 onward", post),
    ("Oct 7+ · precision 1", post[post.geo_precision == 1]),
    ("Oct 7+ · precision 2", post[post.geo_precision == 2]),
    ("Oct 7+ · Explosions/Remote violence", post[post.event_type == "Explosions/Remote violence"]),
    ("Oct 7+ · Violence against civilians", post[post.event_type == "Violence against civilians"]),
    ("Oct 7+ · Battles", post[post.event_type == "Battles"]),
]
imb = pd.DataFrame([imbalance_row(l, d) for l, d in slices]).set_index("Slice")
display(imb.round({"Focal share": 3, "Majority : minority": 2}))

# Weekly view of the same question: how often is a week "quiet" for focal events?
print("Weeks in the Gaza-week table:", len(gaza_week),
      "| weeks with zero political-violence records:", int((gaza_week.political_events == 0).sum()),
      "| weeks with zero focal events:", int((gaza_week.focal_events == 0).sum()))
dw_zero = district_week.assign(zero_focal=district_week.focal_events == 0,
                               zero_political=district_week.political_events == 0)
display(dw_zero.groupby("admin2")[["zero_focal", "zero_political"]].sum().rename(
    columns={"zero_focal": "district-weeks with 0 focal events", "zero_political": "district-weeks with 0 political-violence records"}))

### Chart 1 — How lopsided is each slice, and how big is the smaller class?

In [ ]:
plot_df = imb.iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.25, 1]})

ax = axes[0]
colors = [C_FOCAL if s >= 0.5 else C_GREY for s in plot_df["Focal share"]]
ax.barh(plot_df.index, plot_df["Focal share"], color=colors)
for y, (_, r) in enumerate(plot_df.iterrows()):
    ax.text(r["Focal share"] + 0.012, y, f"{r.Focal:,.0f} / {r.Records:,.0f} = {r['Focal share']:.1%}", va="center", fontsize=8.5)
ax.axvline(0.5, color="black", lw=1, ls="--")
ax.text(0.505, len(plot_df) - 0.45, "balanced (50%)", fontsize=8, va="bottom")
ax.set_xlim(0, 1.3)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Focal share of political-violence records in the slice")
ax.set_ylabel("")
ax.set_title("How lopsided is focal vs. not-focal in each slice of the data?")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)

ax = axes[1]
ax.barh(plot_df.index, plot_df["Minority count"].clip(lower=0.5), color=C_OTHER)
for y, (_, r) in enumerate(plot_df.iterrows()):
    ax.text(max(r["Minority count"], 0.5) * 1.15, y, f"{r['Minority count']:,.0f} ({r['Minority class']})", va="center", fontsize=8.5)
ax.set_xscale("log"); ax.set_xlim(0.5, 2e5)
ax.axvline(100, color="black", lw=1, ls=":")
ax.text(105, len(plot_df) - 0.45, "100 (thin for modelling; illustrative)", fontsize=8, va="bottom")
ax.set_yticklabels([]); ax.set_ylabel("")
ax.set_xlabel("Records in the smaller class (log scale)")
ax.set_title("How many examples does the smaller class have?")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); plt.show()

**Interpretation.** The overall split is nearly even (10,714 focal of 22,753), and so is the period from Oct 7 onward (10,651 of 22,364, ratio 1.10 to 1), but the earlier period is lopsided: 63 focal of 389 records (16.2%, about 5 to 1) with only 63 examples in the smaller class (BQ1, BQ5). Inside the later period the extremes are structural: Violence against civilians is 784 of 790 focal and Battles are 0 of 1,593, so the ratio reaches 131 to 1 and one class is empty. Precision also moves the balance (59.6% focal at precision 1 against 31.7% at precision 2). The event-type extremes are partly built into the focal rule, which needs a civilian actor label, so they describe the coding as much as the events; Task 4 already tests the period and event-type differences, so this chart only shows where the smaller class is too small to model on its own.

**Weekly view.** No Gaza week has zero political-violence records or zero focal events. Only North Gaza has empty district-weeks: 7 with no focal events (6 of them in complete weeks) and 3 with no political-violence records, so shares there are undefined in those weeks. A zero is a zero *recorded* count.

### Suspicious values and outliers

Three kinds of "suspicious": (a) very large fatality totals, (b) weeks whose counts are far from the usual, and (c) data-quality red flags in the files themselves. None of these is deleted; they are flagged for review.

### 4.1 Fatality outliers

**Question.** Fatalities are heavy-tailed with many zeros (Task 1 covers the distribution). Here we ask how far the tail reaches, how much of the reported total sits in it, and when and where it occurs. The review flag in the data marks records with **24 or more** reported fatalities (the lowest flagged value is 24 and the highest unflagged value is 23). Fatalities are reported event totals, not confirmed civilian deaths.

*Source table: `master`. Unit: one event record.*

In [ ]:
f = master.fatalities
thr = int(master.loc[master.high_fatalities_review, "fatalities"].min())
thr_below = int(master.loc[~master.high_fatalities_review, "fatalities"].max())
flag = master[master.high_fatalities_review]
top1 = master.nlargest(int(len(master) * 0.01), "fatalities")

fat_summary = pd.DataFrame({"Value": {
    "Records": len(master),
    "Records with 0 fatalities": f"{(f == 0).sum():,} ({(f == 0).mean():.1%})",
    "Median / 75th / 90th percentile": f"{f.median():.0f} / {f.quantile(.75):.0f} / {f.quantile(.90):.0f}",
    "95th / 99th percentile": f"{f.quantile(.95):.0f} / {f.quantile(.99):.0f}",
    "Maximum": int(f.max()),
    "Mean (pulled up by the tail)": round(f.mean(), 2),
    "IQR upper fence (Q3 + 1.5 x IQR)": round(f.quantile(.75) + 1.5 * (f.quantile(.75) - f.quantile(.25)), 1),
    "Review flag: lowest flagged / highest unflagged": f"{thr} / {thr_below}",
    "Records flagged for review": f"{len(flag):,} ({len(flag) / len(master):.1%} of records)",
    "Share of all reported fatalities in flagged records": f"{flag.fatalities.sum() / f.sum():.1%}",
    "Share of all reported fatalities in the top 1% of records": f"{top1.fatalities.sum() / f.sum():.1%}",
    "Flagged records that are focal": f"{int(flag.focal_event.sum())} of {len(flag)}",
}})
display(fat_summary)

print("Flagged records by event type and focal status")
display(flag.groupby(["event_type", "focal_event"]).size().rename("flagged records").to_frame())
print("Flagged records by location precision:", flag.geo_precision.value_counts().sort_index().to_dict())

### Chart 2 — How long is the fatality tail, and which records sit at the end of it?

In [ ]:
pos = np.sort(f[f > 0].to_numpy())
ecdf = np.arange(1, len(pos) + 1) / len(pos)
top = master.nlargest(12, "fatalities").iloc[::-1]
labels = [f"{d:%b %d, %Y} · {loc}" for d, loc in zip(top.event_date, top.location)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 1.15]})
ax = axes[0]
ax.step(pos, ecdf, where="post", color=C_OTHER)
ax.set_xscale("log")
for xv, lab in [(f.quantile(.95), "95th pct (all records)"), (f.quantile(.99), "99th pct (all records)"), (thr, f"review flag ({thr}+)")]:
    ax.axvline(xv, color=C_FOCAL if "flag" in lab else C_GREY, ls="--", lw=1.2)
    ax.text(xv * 1.05, 0.06 if "flag" in lab else (0.18 if "99" in lab else 0.30), lab, rotation=90, fontsize=8, va="bottom")
ax.set_xlabel("Reported fatalities per record (log scale; records with 0 left out)")
ax.set_ylabel("Share of records with fatalities at or below x")
ax.set_title("How long is the fatality tail among records that report fatalities?")
ax.grid(axis="x", visible=True)

ax = axes[1]
ax.barh(range(len(top)), top.fatalities, color=[C_FOCAL if v else C_GREY for v in top.focal_event])
ax.set_yticks(range(len(top))); ax.set_yticklabels(labels, fontsize=8.5)
for y, v in enumerate(top.fatalities):
    ax.text(v + 4, y, f"{v}", va="center", fontsize=8.5)
ax.set_xlabel("Reported fatalities in one record")
ax.set_title("Which 12 records report the most fatalities? (red = focal, grey = not focal)")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); plt.show()

**Interpretation.** Two thirds of records report zero fatalities (66.1%), the 95th percentile is 12 and the 99th is 39, yet the maximum is 471 (Gaza - Az Zaytun, Oct 17, 2023); the 576 flagged records are only 2.2% of records but hold 37.6% of all reported fatalities (BQ4). The three largest records all fall in Oct to Nov 2023, and 422 of the 576 flagged records are focal, with another 97 being Battles. These records dominate any sum or mean of fatalities, so medians and shares are safer summaries (as Task 2 already does); a very large total may be one event or an aggregated report, which only the event notes can settle.

### Chart 3 — When are the high-fatality records concentrated?

In [ ]:
q = flag.assign(quarter=flag.event_date.dt.to_period("Q").astype(str))
qt = pd.crosstab(q.quarter, q.focal_event).reindex(columns=[True, False], fill_value=0)
qt.columns = ["Focal", "Not focal"]
all_q = master.assign(quarter=master.event_date.dt.to_period("Q").astype(str))
qt = qt.reindex(sorted(all_q.quarter.unique()), fill_value=0)
qt = qt.loc[qt.index >= "2023Q4"]
qt["Share of that quarter's records flagged"] = qt.sum(axis=1) / all_q.groupby("quarter").size().reindex(qt.index)
display(qt.round({"Share of that quarter's records flagged": 3}))

fig, ax = plt.subplots(figsize=(10.5, 4.6))
ax.bar(qt.index, qt["Focal"], color=C_FOCAL, label="Focal")
ax.bar(qt.index, qt["Not focal"], bottom=qt["Focal"], color=C_GREY, label="Not focal")
for x, (_, r) in enumerate(qt.iterrows()):
    ax.text(x, r["Focal"] + r["Not focal"] + 4, f"{int(r['Focal'] + r['Not focal'])}", ha="center", fontsize=9)
ax.set_xlabel("Quarter of event date"); ax.set_ylabel("Records flagged for fatality review")
ax.set_title("When are the high-fatality records concentrated?")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

**Interpretation.** Q4 2023 holds 281 of the 576 flagged records (235 focal, 46 not focal) and 7.7% of that quarter's records are flagged, against 2.6% in Q1 2024 and 1.7% or less in every later quarter (BQ4). The extreme-total problem is therefore mostly a feature of the first three months, so including or dropping flagged records changes early-period results far more than later ones. This is a count threshold, not a verification, and a quiet quarter does not mean low harm; reporting coverage can differ across periods.

### 4.2 Unusual weeks

**Question.** Which Gaza-wide weeks have counts far from the typical week? Two rules are compared on the 103 complete weeks: a **whole-period** robust z-score (median 98 and MAD 17 for focal events) and the **trailing 8-week** rule planned for Phase 4. The first 8 weeks have no trailing score, and the incomplete final week is shown but not scored.

*Source table: `acled_gaza_week.csv`. Unit: one Gaza week.*

In [ ]:
gwc = gaza_week[gaza_week.complete_week].sort_values("week_start").set_index("week_start").copy()
partial = gaza_week[~gaza_week.complete_week]
for c in ["focal_events", "political_events", "focal_share"]:
    gwc[f"rz_{c}"] = robust_z(gwc[c])                    # against the whole period
    gwc[f"tz_{c}"] = trailing_robust_z(gwc[c], window=8)  # against the previous 8 weeks only

med = {c: gwc[c].median() for c in ["focal_events", "political_events", "focal_share"]}
mad = {c: (gwc[c] - med[c]).abs().median() for c in med}
print("Median / MAD of weekly values (complete weeks):",
      {c: (round(med[c], 3), round(mad[c], 3)) for c in med})
print(f"|modified z| > {Z_CUT}:  focal_events {int((gwc.rz_focal_events.abs() > Z_CUT).sum())} weeks | "
      f"political_events {int((gwc.rz_political_events.abs() > Z_CUT).sum())} | focal_share {int((gwc.rz_focal_share.abs() > Z_CUT).sum())}")
print(f"Trailing 8-week rule: focal_events {int((gwc.tz_focal_events.abs() > Z_CUT).sum())} weeks | "
      f"political_events {int((gwc.tz_political_events.abs() > Z_CUT).sum())} | focal_share {int((gwc.tz_focal_share.abs() > Z_CUT).sum())}")

flagged = gwc[(gwc[["rz_focal_events", "rz_political_events", "tz_focal_events", "tz_political_events"]].abs() > Z_CUT).any(axis=1)]
display(flagged[["focal_events", "political_events", "focal_share", "rz_focal_events", "rz_political_events",
                 "tz_focal_events", "tz_political_events"]].round(2))

### Chart 4 — Which weeks are unusual for focal events and for political-violence records?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, col, title, ylab in [
    (axes[0], "focal_events", "Which weeks have unusual numbers of focal events?", "Focal events per week"),
    (axes[1], "political_events", "Which weeks have unusual numbers of political-violence records?", "Political-violence records per week")]:
    s = gwc[col]
    ax.plot(s.index, s, color=C_OTHER, lw=1.3, label="Weekly count (complete weeks)")
    band = 1.4826 * mad[col] * Z_CUT
    ax.axhspan(med[col] - band, med[col] + band, color=C_GREY, alpha=0.18, label=f"Median ± {Z_CUT} robust SD (whole period)")
    ax.axhline(med[col], color=C_GREY, lw=1)
    g_flag = gwc[gwc[f"rz_{col}"].abs() > Z_CUT]
    t_flag = gwc[(gwc[f"tz_{col}"].abs() > Z_CUT) & ~(gwc[f"rz_{col}"].abs() > Z_CUT)]
    ax.scatter(g_flag.index, g_flag[col], color=C_FOCAL, s=45, zorder=3, label="Flagged against the whole period")
    ax.scatter(t_flag.index, t_flag[col], color=C_WARN, marker="D", s=40, zorder=3, label="Flagged only against previous 8 weeks")
    ax.scatter(partial.week_start, partial[col], facecolors="none", edgecolors="black", s=55, zorder=4, label="Incomplete final week (not scored)")
    ax.set_title(title); ax.set_ylabel(ylab)
    ax.legend(frameon=False, ncol=2, fontsize=8, loc="upper right")
axes[1].set_xlabel("Week start")
plt.tight_layout(); plt.show()

**Interpretation.** Against the whole period only three focal-event weeks stand out, all at the very start (Oct 21, Oct 28 with the peak of 254, and Nov 4, 2023), and only five political-violence weeks stand out, all low (Jan 25 to Feb 22, 2025, with 22 to 40 records); no week has an unusual focal *share* (30% to 73%) (BQ4). The trailing rule flags 13 focal-event weeks, but most of the highs (Mar 15 to Apr 5, 2025 and May 17, 2025) are the recovery after the Jan to Feb 2025 collapse, with 90 to 169 events against a long-run median of 98, so a short baseline scores a return to normal as a spike. The files contain no source for the Jan to Feb 2025 fall, so we describe it and do not explain it; the dip in the week of Nov 25, 2023 (58 focal events) is not flagged by either rule, and the first 8 weeks have no trailing score at all.

### District-week check with the same trailing rule

In [ ]:
# Same trailing 8-week rule, applied inside each district (complete weeks only).
dwc = district_week[district_week.complete_week].sort_values(["admin2", "week_start"]).copy()
dwc["tz_focal"] = dwc.groupby("admin2").focal_events.transform(lambda s: trailing_robust_z(s, window=8))
elig = dwc.tz_focal.notna()
dist_flags = (dwc[elig].assign(high=lambda d: d.tz_focal > Z_CUT, low=lambda d: d.tz_focal < -Z_CUT)
              .groupby("admin2").agg(scored_weeks=("tz_focal", "size"), high_weeks=("high", "sum"), low_weeks=("low", "sum")))
dist_flags["flagged share"] = (dist_flags.high_weeks + dist_flags.low_weeks) / dist_flags.scored_weeks
display(dist_flags.round({"flagged share": 3}))
print("All districts together:", int((dist_flags.high_weeks + dist_flags.low_weeks).sum()), "flagged of",
      int(dist_flags.scored_weeks.sum()), "scored district-weeks =",
      f"{(dist_flags.high_weeks + dist_flags.low_weeks).sum() / dist_flags.scored_weeks.sum():.1%}")
print("Highest-scoring district-weeks (upward):")
display(dwc.nlargest(6, "tz_focal")[["admin2", "week_start", "focal_events", "political_events", "tz_focal"]].round({"tz_focal": 2}))

**Interpretation.** In 475 scored district-weeks, 50 (10.5%) are flagged, and the share is highest in North Gaza (14.7%) and Gaza City (13.7%) and lowest in Khan Yunis (7.4%) (BQ4). The six highest upward scores come from March 2025 and one week of July 2024 (Khan Yunis, Jul 20, 41 focal events), which repeats the recovery effect seen above. Flagged weeks are about one in ten, so a weekly anomaly detector faces an imbalanced target of its own, and small districts such as North Gaza have low weekly counts and give noisy scores.

### 4.3 Is the data itself consistent?

In [ ]:
lat_ok = master.latitude.between(31.2, 31.6) & master.longitude.between(34.2, 34.6)
dq = pd.DataFrame([
    ("Duplicate event IDs in the master", int(master.duplicated("event_id_cnty").sum()), "None: no event is counted twice."),
    ("Missing coordinates in the master", int(master[["latitude", "longitude"]].isna().any(axis=1).sum()), "None missing."),
    ("Coordinates outside a rough Gaza box (lat 31.2-31.6, lon 34.2-34.6)", int((~lat_ok).sum()), "All inside: no impossible locations."),
    ("Events with no district (kept in Gaza-wide totals)", int(master.admin2.isna().sum()), "Left out of district tables only."),
    ("Coast-coded events (kept in Gaza-wide totals)", int(master.coastal_location.sum()), "Left out of the land spatial table only."),
    ("Zero-event weeks in the Gaza-week table", int((gaza_week.recorded_events == 0).sum()), "None: weekly activity never stops in the record."),
    ("Incomplete final week", f"{partial.week_start.dt.date.iloc[0]} to {partial.week_end.dt.date.iloc[0]}", "Marked and left out of scoring; the export ends Sep 30, 2025."),
    ("Records with 0 reported fatalities", f"{(master.fatalities == 0).mean():.1%}", "A zero may mean no figure was recorded, not that nobody was hurt."),
    ("Records with time precision other than 1", f"{(master.time_precision != 1).mean():.1%}", "Date is not exact for these; weekly counts can shift by a week."),
    ("Land focal incidents", len(incidents), ""),
    ("Distinct coordinate pairs among them", len(incidents[["latitude", "longitude"]].drop_duplicates()), "Many incidents share one coordinate: they stand for named places, not exact points."),
], columns=["Check", "Result", "Reading"])
display(dq)

**Interpretation.** The checks find no duplicate IDs, no missing coordinates and no coordinates outside the Gaza area, and every Gaza week has recorded activity (BQ5). The cautions are about meaning: 66.1% of records report zero fatalities, 701 events are coast-coded and left out of the land map only, and the 10,583 land focal incidents sit on just **101 distinct coordinates**, so a coordinate stands for a named place and not an exact point. Only 0.8% of records have a time precision other than 1, so weekly counts are stable against date uncertainty.

### Trends

**Question.** Over the two years from Oct 7, 2023, is there a lasting direction in weekly focal events, in political-violence records or in the focal share? Weekly values are summed to months (a week belongs to the month it starts in; complete weeks only) so the long-run shape is easier to see than in the weekly charts of Task 3. A Spearman rank correlation with week number gives a rough direction; the p-values are only a guide because neighbouring weeks are not independent.

*Source table: `acled_gaza_week.csv`. Unit: one Gaza week, summed to months.*

In [ ]:
wk = gwc.reset_index()
wk["month"] = wk.week_start.dt.to_period("M")
mo = wk.groupby("month").agg(focal=("focal_events", "sum"), political=("political_events", "sum"), weeks=("focal_events", "size"))
mo["share"] = mo.focal / mo.political
mo.index = mo.index.astype(str)
display(mo.T.round(3))

wk["quarter"] = wk.week_start.dt.to_period("Q").astype(str)
qtr = wk.groupby("quarter").agg(weeks=("focal_events", "size"), mean_focal_per_week=("focal_events", "mean"),
                                mean_political_per_week=("political_events", "mean"), pooled_share=("focal_events", "sum"))
qtr["pooled_share"] = qtr.pooled_share / wk.groupby("quarter").political_events.sum()
display(qtr.round({"mean_focal_per_week": 1, "mean_political_per_week": 1, "pooled_share": 3}))

# Rank trend against time (exploratory: weeks are serially dependent, so the p-values are only a rough guide)
t = np.arange(len(gwc))
trend = []
for c in ["focal_events", "political_events", "focal_share"]:
    rho, p = spearmanr(t, gwc[c])
    half = len(gwc) // 2
    trend.append({"Weekly measure": c, "Spearman rho vs week number": rho, "p (rough)": p,
                  "Median, first half": gwc[c].iloc[:half].median(), "Median, second half": gwc[c].iloc[half:].median()})
display(pd.DataFrame(trend).set_index("Weekly measure").round(3))

### Chart 5 — Is there a lasting trend in monthly focal events and in their share?

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(mo))
ax.bar(x, mo.focal, color=C_FOCAL, alpha=0.85, label="Focal events (sum of complete weeks)")
ax.set_xticks(x); ax.set_xticklabels(mo.index, rotation=60, ha="right", fontsize=8.5)
ax.set_ylabel("Focal events per month"); ax.set_xlabel("Month of week start")
ax.set_title("Is there a lasting trend in monthly focal events and in their share?")
ax2 = ax.twinx()
ax2.plot(x, mo["share"], color=C_OTHER, marker="o", lw=1.8, label="Focal share of political-violence records")
ax2.set_ylabel("Focal share"); ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax2.set_ylim(0, 0.8)
ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, frameon=False, loc="upper right")
plt.tight_layout(); plt.show()

**Interpretation.** Focal events show no lasting direction (Spearman 0.10, median 96 per week in the first half and 101 in the second), but political-violence records fall (rho -0.55, median 253 to 200) and the focal share rises (rho 0.51, median 38.3% to 54.3%), from a trough of 34.0% in Jan 2024 to 67.0% in Sep 2025 (BQ1, BQ4). In other words the share rises mainly because the non-focal part of political violence falls while focal counts hold, and the monthly counts swing from 755 (Oct 2023) down to 73 (Feb 2025) and back to 708 (May 2025), so the series is a set of level shifts and not a straight line. The first and last months cover only 4 and 3 weeks, and the rank-correlation p-values ignore serial dependence.

### Seasonality and persistence

**Question.** Do the same calendar months or weekdays repeat, and does a week resemble the weeks before it? The data cover about two years, so each calendar month appears at most twice (Oct to Dec in 2023 and 2024, Jan to Sep in 2024 and 2025) and a 52-week cycle cannot be tested at all. We therefore compare the two years month by month, look at weekdays, and use the autocorrelation function (ACF) to check persistence.

*Source tables: `acled_gaza_week.csv` (months, ACF) and `acled_events_clean.csv` (weekdays, focal events from Oct 7).*

### Chart 6 — Do the same calendar months and weekdays repeat?

In [ ]:
wk["year"], wk["mon"] = wk.week_start.dt.year, wk.week_start.dt.month
moy = wk.groupby(["year", "mon"]).focal_events.mean().unstack(0)
display(moy.round(1))
both = moy[[2024, 2025]].dropna()
rho_yy, _ = spearmanr(both[2024], both[2025])
print(f"Months present in both 2024 and 2025: {len(both)} (Jan-Sep). Spearman rho between the two years' monthly means = {rho_yy:.2f}")

post_focal = master[(master.focal_event) & (master.period == AFTER)]
dow = post_focal.event_date.dt.dayofweek.value_counts().sort_index()
dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print("Focal events by weekday:", dow.to_dict(), f"| highest / lowest = {dow.max() / dow.min():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
for yr, col in zip([2023, 2024, 2025], [C_GREY, C_OTHER, C_FOCAL]):
    s = moy[yr].dropna()
    ax.plot(s.index, s.values, marker="o", color=col, lw=1.8, label=str(yr) + (" (Oct-Dec only)" if yr == 2023 else " (Jan-Sep only)" if yr == 2025 else ""))
ax.set_xticks(range(1, 13)); ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_xlabel("Calendar month"); ax.set_ylabel("Mean focal events per complete week")
ax.set_title("Do the same calendar months repeat across years?")
ax.legend(frameon=False)
ax = axes[1]
ax.bar(dow.index, dow.values, color=C_OTHER)
ax.axhline(dow.mean(), color="black", ls="--", lw=1)
for i, v in enumerate(dow.values):
    ax.text(i, v + 15, f"{v:,}", ha="center", fontsize=8.5)
ax.set_ylim(0, dow.max() * 1.12)
ax.set_xlabel("Day of week (event date)"); ax.set_ylabel("Focal events, Oct 7, 2023 onward")
ax.set_title("Is any weekday busier for recorded focal events?")
plt.tight_layout(); plt.show()

*(The cell above also prints the monthly means and weekday counts behind the chart.)*

**Interpretation.** The two years do not share a calendar shape: for the nine months present in both (Jan to Sep), the correlation between 2024 and 2025 monthly means is -0.30, February is 105.5 per week in 2024 and 18.2 in 2025, and Oct to Dec 2023 (188.8, 151.5 and 124.6) is far above the same months of 2024 (about 91) (BQ4). Weekday counts range only from 1,444 (Sunday) to 1,597 (Tuesday), a ratio of 1.11, so no day stands out. There is no evidence of seasonality here, but two years cannot rule out a longer cycle, and weekday counts also reflect when events are reported.

### Chart 7 — Does a week resemble the weeks before it?

In [ ]:
def acf(x, nlags):
    x = np.asarray(x, float); x = x - x.mean()
    return np.array([(x[:-k] * x[k:]).sum() / (x * x).sum() for k in range(1, nlags + 1)])

series = gwc.focal_events.to_numpy(float)
lags = np.arange(1, 13)
acf_level, acf_diff = acf(series, 12), acf(np.diff(series), 12)
band = 1.96 / np.sqrt(len(series))
print("ACF of weekly focal events, lags 1-12:", np.round(acf_level, 2))
print("ACF of week-to-week changes, lags 1-12:", np.round(acf_diff, 2))
print(f"Approximate 95% band: ±{band:.2f}  (n = {len(series)} weeks).")
print("A 52-week cycle cannot be checked: the table spans only", len(gaza_week), "weeks (about two years).")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
for ax, vals, title in [(axes[0], acf_level, "Does a week resemble the weeks before it? (weekly focal events)"),
                        (axes[1], acf_diff, "After removing the level: do week-to-week changes repeat?")]:
    ax.bar(lags, vals, color=C_OTHER, width=0.6)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhspan(-band, band, color=C_GREY, alpha=0.25, label="≈95% band if no autocorrelation")
    ax.set_xlabel("Lag (weeks)"); ax.set_title(title, fontsize=10)
    ax.set_xticks(lags)
axes[0].set_ylabel("Autocorrelation"); axes[0].legend(frameon=False)
plt.tight_layout(); plt.show()

**Interpretation.** Weekly focal events are strongly persistent: the autocorrelation is 0.76 at lag 1 and decays slowly (0.63, 0.49, 0.39, 0.34, 0.26) until it falls inside the ±0.19 band at lag 7, while the week-to-week *changes* are essentially uncorrelated (only lag 1, -0.26, sits just outside the band) (BQ4). A slow decay with no peak at lag 4 or 8 points to level shifts (the regimes in Section 7) and not to a repeating cycle, which supports using the previous week and the previous four weeks as history features in Phase 4. Level shifts inflate the autocorrelation of any non-stationary series, so these values describe persistence, not predictability.

### Segments and exploratory clustering

Three groupings: districts (by volume and burstiness), weeks (by what kind of week they were) and places (by where incidents are recorded). The last one also previews the DBSCAN settings Phase 4 will use. These are **descriptive groupings**, not validated models, and a cluster is a recorded concentration, not an exact attack point.

### 7.1 District segments

**Question.** Which districts carry most focal events, is that split stable, and which districts are bursty? Only the 5 known districts are used (the 4 events with no district are excluded, 3 of them focal), and the incomplete final week is dropped. Burstiness is the standard deviation divided by the mean of weekly counts.

*Source table: `acled_district_week.csv`. Unit: one district-week.*

### Chart 8 — Which districts carry the focal events, and how steady are they?

In [ ]:
pivot = dwc.pivot(index="week_start", columns="admin2", values="focal_events")
roll = pivot.rolling(4).sum()
share4 = roll.div(roll.sum(axis=1), axis=0).dropna()
order = pivot.sum().sort_values(ascending=False).index.tolist()

prof = dwc.groupby("admin2").focal_events.agg(total="sum", median="median", mean="mean", sd="std", peak="max")
prof["cv"] = prof.sd / prof["mean"]
prof["share_of_known_district_focal"] = prof.total / prof.total.sum()
prof["weeks_with_zero"] = dwc.groupby("admin2").focal_events.apply(lambda s: int((s == 0).sum()))
prof = prof.loc[order]
display(prof.round({"mean": 1, "sd": 1, "cv": 2, "share_of_known_district_focal": 3}))
print("Range of each district's 4-week share of known-district focal events:")
display(pd.DataFrame({"min": share4.min(), "max": share4.max()}).loc[order].round(3))

dist_colors = dict(zip(order, ["#8D3B42", "#315A7D", "#E0A458", "#6B8F71", "#9AA5B1"]))
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
ax.stackplot(share4.index, [share4[d] for d in order], labels=order, colors=[dist_colors[d] for d in order], alpha=0.9)
ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.set_xlim(share4.index.min(), share4.index.max())
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8.5)
ax.set_xlabel("Week start (4-week window ending that week)"); ax.set_ylabel("Share of focal events with a known district")
ax.set_title("Which districts carry the focal events, and does the split change over time?")
ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.14), fontsize=8.5)
ax = axes[1]
ax.scatter(prof["median"], prof["cv"], s=prof.total / 6, color=[dist_colors[d] for d in prof.index], alpha=0.9, edgecolor="black")
for d, r in prof.iterrows():
    ax.annotate(d, (r["median"], r["cv"]), textcoords="offset points", xytext=(9, 7), fontsize=9)
ax.set_xlim(0, prof["median"].max() * 1.3); ax.set_ylim(0, prof.cv.max() * 1.2)
ax.set_xlabel("Median focal events per week (typical volume)"); ax.set_ylabel("Variability (sd / mean of weekly counts)")
ax.set_title("Typical volume vs. burstiness by district (bubble = total)")
ax.grid(axis="x", visible=True)
plt.tight_layout(); plt.show()

**Interpretation.** Gaza City carries the most focal events (3,411, 32.3% of those with a known district; median 30 per week and a peak of 102), followed by Khan Yunis (21.6%) and Deir El Balah (21.4%), while North Gaza (12.5%) and Rafah (12.3%) are each about 40% of Gaza City's total (BQ4). North Gaza is the most bursty (variability 0.83, with 6 complete weeks of zero focal events) and Deir El Balah the steadiest (0.40), and a district's 4-week share swings widely, for example Gaza City from 15.1% to 55.8% and Rafah from 2.4% to 47.0%. Districts therefore need their own baselines and not one Gaza-wide threshold; these shares are of events with a known district, and a district count can rise because a place is better documented.

### 7.2 Segments of weeks

**Question.** Do weeks fall into a few kinds? We cluster the 103 complete weeks with k-means on three standardised measures (log of focal events, log of political-violence records, focal share). The silhouette score helps choose *k* and is only a rough guide with 103 points.

In [ ]:
feat = gwc[["focal_events", "political_events", "focal_share"]].copy()
feat["focal_events"], feat["political_events"] = np.log1p(feat.focal_events), np.log1p(feat.political_events)
Xs = StandardScaler().fit_transform(feat)

sil = []
for k in range(2, 6):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Xs)
    sil.append({"k": k, "silhouette": silhouette_score(Xs, km.labels_), "smallest cluster (weeks)": int(np.bincount(km.labels_).min())})
display(pd.DataFrame(sil).set_index("k").round(3))

`k = 2` has the best silhouette (0.67) but only separates 9 low weeks from everything else. We use **k = 3**, which keeps those low weeks as their own group and also splits the rest by focal share (silhouette 0.51, a moderate structure); the choice is for description, not a claim that three is the true number.

### Chart 9 — What kinds of weeks occur, and when?

In [ ]:
K = 3
km = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit(Xs)
gwc["cluster"] = km.labels_
# Name clusters by their median focal count so the labels are stable between runs
rank = gwc.groupby("cluster").focal_events.median().sort_values().index.tolist()
names = {c: f"Group {chr(65 + i)}" for i, c in enumerate(rank)}
gwc["regime"] = gwc.cluster.map(names)
cl_prof = gwc.reset_index().groupby("regime").agg(weeks=("focal_events", "size"), first=("week_start", "min"), last=("week_start", "max"),
                                                  median_focal=("focal_events", "median"), median_political=("political_events", "median"),
                                                  median_share=("focal_share", "median"))
display(cl_prof.round({"median_focal": 1, "median_political": 1, "median_share": 3}))
print("Week starts per group:")
for g, d in gwc.groupby("regime"):
    print(f"  {g}:", ", ".join(f"{x:%Y-%m-%d}" for x in d.index) if len(d) <= 12 else f"{len(d)} weeks; first {d.index.min():%Y-%m-%d}, last {d.index.max():%Y-%m-%d}")

pal = dict(zip(sorted(names.values()), [C_GREY, C_OTHER, C_FOCAL]))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.9), gridspec_kw={"width_ratios": [1, 1.5]})
ax = axes[0]
for g, d in gwc.groupby("regime"):
    ax.scatter(d.political_events, d.focal_events, color=pal[g], s=36, alpha=0.9, label=f"{g} ({len(d)} weeks)")
ax.set_xlabel("Political-violence records per week"); ax.set_ylabel("Focal events per week")
ax.set_title("Which kinds of weeks group together?"); ax.legend(frameon=False); ax.grid(axis="x", visible=True)
ax = axes[1]
ax.plot(gwc.index, gwc.focal_events, color="black", lw=0.7, alpha=0.5)
for g, d in gwc.groupby("regime"):
    ax.scatter(d.index, d.focal_events, color=pal[g], s=36, label=g, zorder=3)
ax.set_xlabel("Week start"); ax.set_ylabel("Focal events per week")
ax.set_title("When does each kind of week occur?"); ax.legend(frameon=False, ncol=3)
plt.tight_layout(); plt.show()

**Interpretation.** Group A is 8 weeks (Jan 18 to Mar 8, 2025) with a median of 21.5 focal events and 39 political-violence records, Group B is 51 weeks (Nov 2023 to Dec 2024) with a median of 92 focal events and a 38.3% share, and Group C is 44 weeks with a median of 123.5 focal events and a 57.4% share, made up of the first weeks after Oct 7, 2023, 6 later weeks of 2024, 2 weeks of early January 2025 and 28 weeks from Mar 15, 2025 on (BQ4). The groups follow time more than chance: an early high phase, a long 2024 plateau, the Jan to Mar 2025 gap, and a high-share 2025 phase. The B and C boundary is blurry (14 weeks of C fall before 2025), the groups are descriptive labels, and with 103 weeks a different seed or feature set could move some weeks across.

### 7.3 Where are incidents recorded?

**Question.** Where are post-Oct 7 land focal incidents concentrated, and does the picture survive when only precision-1 records are kept (BQ3, BQ5)? Many incidents share the same coordinate, so we first aggregate to distinct coordinate pairs ("sites") and then measure how unevenly incidents spread over them (Gini: 0 is even, 1 is all in one place).

*Source table: `acled_focal_incidents.csv` (land only; coast-coded events excluded). Unit: one incident, then one coordinate pair.*

In [ ]:
def site_table(df):
    return (df.groupby(["latitude", "longitude"])
              .agg(events=("event_id_cnty", "size"), district=("admin2", lambda s: s.mode().iat[0]),
                   location=("location", lambda s: s.mode().iat[0]), precision1_share=("geo_precision_1", "mean"))
              .reset_index().sort_values("events", ascending=False))

sites_full = site_table(incidents)
inc_p1 = incidents[incidents.geo_precision_1]
sites_p1 = site_table(inc_p1)

def gini(v):
    v = np.sort(np.asarray(v, float)); n = len(v)
    return (2 * np.arange(1, n + 1) - n - 1).dot(v) / (n * v.sum())

conc = pd.DataFrame({
    "Incidents": [len(incidents), len(inc_p1)],
    "Distinct coordinate pairs": [len(sites_full), len(sites_p1)],
    "Median incidents per coordinate": [sites_full.events.median(), sites_p1.events.median()],
    "Top 10 coordinates, share of incidents": [sites_full.events.head(10).sum() / len(incidents), sites_p1.events.head(10).sum() / len(inc_p1)],
    "Top 20 coordinates, share of incidents": [sites_full.events.head(20).sum() / len(incidents), sites_p1.events.head(20).sum() / len(inc_p1)],
    "Gini of incidents across coordinates": [gini(sites_full.events), gini(sites_p1.events)],
}, index=["All land focal incidents", "Precision 1 only"]).T
display(conc.round(3))
print("Share of incidents by district:")
display(incidents.admin2.value_counts(normalize=True).round(3).rename("share").to_frame())
print("Ten largest coordinates (all land focal incidents):")
display(sites_full.head(10).round({"precision1_share": 3}))

### Chart 10 — Where are land focal incidents recorded, full vs. precision-1?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.8), sharex=True, sharey=True)
vmax = sites_full.events.max()
for ax, sites, title in [(axes[0], sites_full, f"All land focal incidents ({len(incidents):,})"),
                         (axes[1], sites_p1, f"Precision 1 only ({len(inc_p1):,})")]:
    s = sites.sort_values("events")
    sc = ax.scatter(s.longitude, s.latitude, s=6 + s.events * 0.3, c=s.events, cmap="Reds", norm=LogNorm(vmin=1, vmax=vmax),
                    edgecolor="black", linewidth=0.4, alpha=0.85)
    ax.set_title(title); ax.set_xlabel("Longitude")
    ax.set_aspect(1 / np.cos(np.radians(31.45))); ax.grid(True)
    for _, r in sites.head(4).iterrows():
        ax.annotate(r.location, (r.longitude, r.latitude), textcoords="offset points", xytext=(10, 8), fontsize=7.5,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8),
                    arrowprops=dict(arrowstyle="-", lw=0.6, color="black"))
axes[0].set_ylabel("Latitude")
fig.suptitle("Where are land focal incidents recorded, and does the picture hold when only precision-1 records are kept?", fontsize=11, fontweight="bold")
cb = fig.colorbar(sc, ax=axes, shrink=0.75, pad=0.02); cb.set_label("Incidents at the coordinate (log scale)")
plt.show()

**Interpretation.** The 10,583 incidents sit on only 101 coordinate pairs along a narrow north-east to south-west strip, with the largest bubbles in Gaza City (Az Zaytun, Sheikh Radwan, Shujaiyya), Jabalya, and the central towns of An Nusayrat, Al Burayj and Deir al Balah (BQ3). The precision-1 map (7,576 incidents, 95 coordinates) keeps the same overall shape but loses 6 small coordinates (41 incidents) and visibly thins the southern part of the strip (BQ5). There is no basemap and coordinates repeat for named places, so this shows recorded concentrations and not exact attack points or borders.

### Chart 11 — Which coordinates hold the most incidents, and how unevenly are incidents spread?

In [ ]:
top12 = sites_full.head(12).iloc[::-1].copy()
top12["p1_events"] = (top12.events * top12.precision1_share).round()
top12["other_events"] = top12.events - top12.p1_events

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
ax.barh(top12.location, top12.p1_events, color=C_FOCAL, label="Precision 1")
ax.barh(top12.location, top12.other_events, left=top12.p1_events, color=C_GREY, label="Precision 2-3")
for y, (_, r) in enumerate(top12.iterrows()):
    ax.text(r.events + 6, y, f"{int(r.events)}  ({r.precision1_share:.0%} precision 1)", va="center", fontsize=8.5)
ax.set_xlim(0, top12.events.max() * 1.4)
ax.set_xlabel("Land focal incidents at the coordinate"); ax.set_title("Which coordinates hold the most incidents, and how precise are they?")
ax.legend(frameon=False, loc="lower right"); ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)

ax = axes[1]
for sites, lab, col in [(sites_full, "All land focal incidents", C_FOCAL), (sites_p1, "Precision 1 only", C_OTHER)]:
    v = np.sort(sites.events.to_numpy())
    ax.plot(np.r_[0, np.arange(1, len(v) + 1) / len(v)], np.r_[0, v.cumsum() / v.sum()], color=col, lw=2, label=f"{lab} (Gini {gini(v):.2f})")
ax.plot([0, 1], [0, 1], color="black", ls="--", lw=1, label="Even spread")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Share of coordinates (smallest to largest)"); ax.set_ylabel("Cumulative share of incidents")
ax.set_title("How unevenly are incidents spread across coordinates?"); ax.legend(frameon=False, loc="upper left"); ax.grid(True)
plt.tight_layout(); plt.show()

**Interpretation.** The 10 busiest coordinates hold 33.8% of all incidents and the top 20 hold 56.5% (Gini 0.57); with precision 1 only, these rise to 38.1% and 62.5% (Gini 0.61), so the restriction sharpens concentration (BQ3, BQ5). Among the 12 largest coordinates the precision-1 share runs from 38% to 94% (Tal as Sultan and Southern Remal are lowest, Deir al Balah highest), and 30 of the 101 coordinates are below 50% precision 1, so the precision filter removes places unevenly. A busy coordinate may simply be a well-documented place name, so concentration here is a statement about the records.

### 7.4 Preview of DBSCAN settings (for Phase 4)

**Question.** Phase 4 plans DBSCAN on land focal coordinates. Before it starts, how sensitive is the result to the radius and to `min_samples`, and does the precision-1 subset behave differently? Following the project rule, coordinates are converted to **radians** and the **haversine** distance is used (latitude and longitude are *not* standardised separately). This is a sensitivity preview, not a tuned model: there are no labels to validate against, and repeated coordinates matter because `min_samples` counts repeated incidents, not distinct places.

In [ ]:
R_EARTH_KM = 6371.0088
def run_dbscan(df, eps_km, min_samples):
    X = np.radians(df[["latitude", "longitude"]].to_numpy())          # geographic coordinates in radians
    db = DBSCAN(eps=eps_km / R_EARTH_KM, min_samples=min_samples, metric="haversine", algorithm="ball_tree").fit(X)
    return db.labels_

rows = []
for subset, d in [("All land focal incidents", incidents), ("Precision 1 only", inc_p1)]:
    for ms in (10, 50):
        for eps in (0.25, 0.5, 1.0, 1.5, 2.0, 3.0):
            lab = run_dbscan(d, eps, ms)
            n_cl = len(set(lab)) - (1 if -1 in lab else 0)
            counts = pd.Series(lab[lab >= 0]).value_counts()
            rows.append({"subset": subset, "min_samples": ms, "eps_km": eps, "clusters": n_cl,
                         "noise_share": (lab == -1).mean(), "largest_cluster_share": counts.max() / len(d) if len(counts) else 0.0})
db_grid = pd.DataFrame(rows)
display(db_grid.round({"noise_share": 3, "largest_cluster_share": 3}))

### Chart 12 — How do the number of clusters, noise and the largest cluster change with the radius?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
styles = {("All land focal incidents", 10): (C_FOCAL, "-"), ("All land focal incidents", 50): (C_FOCAL, "--"),
          ("Precision 1 only", 10): (C_OTHER, "-"), ("Precision 1 only", 50): (C_OTHER, "--")}
for ax, col, ylab, title in [(axes[0], "clusters", "Clusters found", "How many clusters at each radius?"),
                             (axes[1], "noise_share", "Share of incidents left as noise", "How much is left unclustered?"),
                             (axes[2], "largest_cluster_share", "Share of incidents in the largest cluster", "When does everything merge?")]:
    for (subset, ms), (c, ls) in styles.items():
        d = db_grid[(db_grid.subset == subset) & (db_grid.min_samples == ms)]
        ax.plot(d.eps_km, d[col], color=c, ls=ls, marker="o", label=f"{subset.replace(' land focal incidents', '')}, min_samples={ms}")
    if col != "clusters":
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("DBSCAN radius (km, haversine)"); ax.set_ylabel(ylab); ax.set_title(title, fontsize=10)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

**Interpretation.** Radii of 0.25 and 0.5 km give identical results (81 clusters with `min_samples` 10) because the recorded coordinates are discrete places spaced further apart; the count then drops to 58 at 1 km, 30 at 1.5 km and 19 at 2 km, where the largest cluster already holds 42.4% of incidents, and at 3 km everything merges into a single cluster (BQ3). Raising `min_samples` to 50 leaves more incidents as noise (7.0% of all incidents, 10.8% at precision 1, at 0.5 km), and the precision-1 subset gives fewer clusters at every radius below 3 km (for example 69 against 81 at 0.5 km). A usable radius for Phase 4 therefore lies roughly between 1 and 2 km, and because 81 of the 101 coordinates already hold 10 or more incidents, DBSCAN here mostly groups named places, so Phase 4 should compare it with the district counts and consider weighting by incidents per coordinate.

### Pattern register

The register collects what Sections 3 to 7 found, with the business question and what each pattern means for the next tasks. All numbers are repeated from the sections above.

In [ ]:
register = pd.DataFrame([
    ("Imbalance", "Focal share is 16.2% before Oct 7 (63 of 389) and 47.6% after (10,651 of 22,364); Battles are 0 of 1,593 and Violence against civilians 784 of 790 after Oct 7", "BQ1, BQ5", "The earlier period and the extreme event types have too few minority cases to model on their own"),
    ("Suspicious values", "576 records report 24+ fatalities (2.2% of records) and hold 37.6% of all reported fatalities; 281 of them fall in Q4 2023", "BQ4", "Never delete them; run results with and without them"),
    ("Suspicious values", "13 complete weeks are flagged for focal events by a trailing 8-week median/MAD rule: lows in Mar and Jun 2024, Jan to Feb 2025 and Jun 2025, and highs (Mar 15 to Apr 5 and May 17, 2025) that mostly follow the early-2025 collapse", "BQ4", "A short trailing window scores a recovery as a spike; test a longer window"),
    ("Suspicious values", "50 of 475 scored district-weeks (10.5%) are flagged, most often in North Gaza (14.7%) and Gaza City (13.7%)", "BQ4", "The anomaly target is imbalanced; small districts are noisy"),
    ("Trend", "Weekly focal events show no lasting direction (rho 0.10), political-violence records fall (rho -0.55) and the focal share rises (rho 0.51; 34.0% in Jan 2024 to 67.0% in Sep 2025)", "BQ1, BQ4", "Use counts and shares as separate signals"),
    ("Seasonality", "No repeating calendar pattern: the monthly means of 2024 and 2025 are not aligned (rho -0.30) and no weekday stands out (ratio 1.11)", "BQ4", "Do not add seasonal terms; two years cannot test a yearly cycle"),
    ("Persistence", "Weekly focal events are autocorrelated (0.76 at lag 1, above the 95% band through lag 6) while week-to-week changes are not", "BQ4", "Supports previous-week and previous-four-week features"),
    ("Segments", "Gaza City holds 32.3% of known-district focal events; North Gaza is low in volume but the most bursty (0.83)", "BQ4", "Use per-district baselines, not one Gaza-wide threshold"),
    ("Clustering of weeks", "Three groups of weeks: 8 low weeks (Jan 18 to Mar 8, 2025), 51 weeks of a 2024 plateau and 44 higher-share weeks (the first weeks after Oct 7, 2023 and Mar to Sep 2025)", "BQ4", "Consider a regime indicator and treat the early weeks and the early-2025 gap with care"),
    ("Spatial concentration", "10,583 incidents sit on 101 distinct coordinates; the top 10 hold 33.8% and the top 20 hold 56.5%", "BQ3", "DBSCAN mostly groups named places; consider weighting by incidents per coordinate"),
    ("Precision", "Precision-1 keeps 95 of the 101 coordinates, but among the 12 largest coordinates its share runs from 38% to 94%; concentration rises (Gini 0.57 to 0.61)", "BQ3, BQ5", "Run DBSCAN on both the full and precision-1 subsets"),
    ("DBSCAN preview", "Clusters fall from 81 at 0.5 km to 19 at 2 km and merge into one at 3 km (min_samples 10)", "BQ3", "Try radii of about 1 to 2 km"),
], columns=["Pattern type", "What we saw", "Business question", "Phase 4 implication"])
with pd.option_context("display.max_colwidth", None):
    display(register)

# Task 6 — Answer or refine the Phase 1 business questions

The earlier answer draft contains some subtotals that differ from the analysis contribution. The tables below recompute supporting values from the loaded Phase 2 CSVs. Use the displayed counts as the evidence base for the answers.

### BQ1 — Before/after counts and focal shares

How do recorded focal-event counts and their share of political-violence events differ before and after October 7, 2023 in Gaza?

In [ ]:
bq1=political.groupby("period").agg(political_records=("event_id_cnty","size"),focal_events=("focal_event","sum"))
bq1["focal_share"]=bq1.focal_events/bq1.political_records
bq1["focal_events_per_week"]=bq1.focal_events*7/bq1.index.map(PERIOD_DAYS)
display(bq1.reindex([BEFORE,AFTER]).round(3))

**Answer:** Recorded focal-event volume and focal share are higher after October 7. Period lengths differ, so the weekly rate is included. The earlier Gaza period is sparse and should be treated as descriptive context. See Task 2's period comparison and Task 4 Hypothesis A; the association does not establish that the cutoff caused the change.

### BQ2 — Actors and event/sub-event types

Which recorded actors and event/sub-event types recur among focal events, and which roles can be supported by coding and event notes?

In [ ]:
focal_events_df=master.loc[master.focal_event].copy()
print("Event types among focal events:"); display(focal_events_df.event_type.value_counts().rename("count").to_frame())
print("Top sub-event types:"); display(focal_events_df.sub_event_type.value_counts().head(10).rename("count").to_frame())
print("Top actor1 labels:"); display(focal_events_df.actor1.value_counts().head(10).rename("count").to_frame())
print("Top actor2 labels:"); display(focal_events_df.actor2.value_counts(dropna=False).head(10).rename("count").to_frame())
print("Civilians (Palestine) in actor1:",int(focal_events_df.actor1.eq("Civilians (Palestine)").sum()))
print("Civilians (Palestine) in actor2:",int(focal_events_df.actor2.eq("Civilians (Palestine)").sum()))

**Answer:** Focal records concentrate in a small number of recorded event and sub-event categories. Actor names reflect ACLED coding and do not by themselves verify who performed an action. Review `notes` and `source` before assigning roles. See Task 2's actor/event cross-tab.

### BQ3 — Geographic concentrations and precision

Where do post–October 7 land focal events concentrate, and how does precision affect the view?

In [ ]:
district_concentration=incidents.groupby("admin2").size().sort_values(ascending=False).rename("land_focal_incidents").to_frame()
district_concentration["share"]=district_concentration.land_focal_incidents/len(incidents)
site_counts=incidents.groupby(["latitude","longitude"]).size().sort_values(ascending=False)
p1=incidents.loc[incidents.geo_precision_1]
p1_sites=p1.groupby(["latitude","longitude"]).size().sort_values(ascending=False)
bq3=pd.DataFrame({"all_land_focal":[len(incidents),incidents[["latitude","longitude"]].drop_duplicates().shape[0],site_counts.head(10).sum()/len(incidents),site_counts.head(20).sum()/len(incidents)],
 "precision_1_only":[len(p1),p1[["latitude","longitude"]].drop_duplicates().shape[0],p1_sites.head(10).sum()/len(p1),p1_sites.head(20).sum()/len(p1)]},
 index=["Incidents","Distinct coordinate pairs","Top 10 coordinate share","Top 20 coordinate share"])
display(district_concentration.round(3)); display(bq3.round(3))

**Answer:** Use these computed district/coordinate summaries and Task 5's matched maps to describe which concentrations persist under the precision-1 restriction and how much data it excludes. Repeated coordinates represent recorded locations, not necessarily exact incident points.

### BQ4 — High or sustained district-weeks

Which district-weeks show unusually high activity or sustained increases relative to earlier local observations?

In [ ]:
complete_dw=district_week.loc[district_week.complete_week].copy()
max_gaza=gaza_week.loc[gaza_week.focal_events.idxmax(),["week_start","focal_events","political_events","focal_share"]]
max_dist=complete_dw.loc[complete_dw.focal_events.idxmax(),["admin2","week_start","focal_events","political_events","focal_share"]]
print("Maximum Gaza-wide week:"); display(max_gaza.to_frame("value"))
print("Maximum district-week:"); display(max_dist.to_frame("value"))
totals=complete_dw.groupby("admin2").agg(focal_events=("focal_events","sum"),political_events=("political_events","sum"))
totals["focal_share"]=totals.focal_events/totals.political_events
display(totals.sort_values("focal_events",ascending=False))

**Answer:** The tables show the highest Gaza-wide week, the highest district-week, and district totals. Task 5's trailing-baseline analysis indicates why district-specific history matters. Weekly shares can be unstable with small denominators; zero means zero recorded activity only.

### BQ5 — Sensitivity to period, geography, precision, and baseline

Which patterns change under alternate cohorts, geography, precision restrictions, and baselines?

In [ ]:
precision_summary=(political.groupby(["period","geo_precision"],observed=True)
 .agg(records=("event_id_cnty","size"),focal=("focal_event","sum")))
precision_summary["focal_share"]=precision_summary.focal/precision_summary.records
display(precision_summary.round(3))
print("Gaza-week focal total minus district-week focal total:",int(gaza_week.focal_events.sum()-district_week.focal_events.sum()))
print("Incomplete week:",gaza_week.loc[~gaza_week.complete_week,"week_start"].dt.date.tolist())

**Answer:** The precision-stratified table and matched spatial views show the main quality-related sensitivity. District totals omit records with unknown districts by design. Report whether each finding persists under the alternative precision subset and baseline; do not treat a zero-record week as absence of harm.

# Task 7 — Modeling implications
## Useful features, promising techniques, and issues to handle

Phase 1 proposed DBSCAN for geographic concentration and weekly anomaly detection using a trailing median/MAD baseline compared with Isolation Forest. The choices below connect that plan to observed EDA evidence.

### Candidate features and technique decisions

| Method / question | EDA evidence | Candidate inputs | Phase 4 recommendation |
|---|---|---|---|
| Spatial concentration: DBSCAN | Task 5 maps and parameter sweep; coordinate repetition and precision sensitivity | Post–October 7 land focal coordinates in radians with Haversine distance | Compare radii around 1-2 km and multiple `min_samples`; run full and precision-1 subsets; compare with district summaries. Interpret as concentrations of recorded locations. |
| Weekly anomaly detection: trailing median/MAD | Persistent weekly counts, a shared low stretch, and short-window recovery flags | District-week counts and eligible prior-week history | Keep an explainable trailing baseline; test longer windows or a recovery-aware rule because short windows can call recovery an anomaly. |
| Weekly multivariate anomaly: Isolation Forest | Weekly counts/shares differ by district; some measures are correlated or derived | Features known before the week being assessed | Compare against the simple baseline using chronological evaluation; exclude current-week outcomes as predictors. |
| Sensitivity | Precision-1 and precision-2 shares differ; district totals are lower by construction | Precision-restricted and complete-week subsets | Report full and restricted results together; do not impute unknown districts or structural actor blanks. |

### Modeling risks surfaced by EDA

- **Class imbalance:** focal prevalence varies sharply across periods and event types; some groups have an empty class. Do not rely on accuracy alone.
- **Multicollinearity/redundancy:** total counts, political counts, focal counts, shares, and rolling sums may encode overlapping information.
- **Temporal leakage:** use only prior-week history for an alert and evaluate chronologically.
- **Spatial measurement:** repeated coordinates represent recorded locations; precision restriction removes records unevenly.
- **Reporting bias:** event-volume changes may reflect documentation coverage or coding as well as recorded activity.

### Phase 4 handoff checklist
- [ ] Keep clustering and anomaly detection as different technique families.
- [ ] Compare each technique to its stated baseline and record settings.
- [ ] Use features available at decision time; avoid target leakage.
- [ ] Evaluate chronologically and note exploratory-period reuse.
- [ ] Treat clusters and alerts as leads for source review, not proof of cause or intent.

## Limitations, reproducibility, and sources

### Limitations
- ACLED may underreport events; documentation coverage can change by time and place.
- Fatalities are event totals and do not automatically identify civilian deaths.
- Actor labels, flags, locations, and precision reflect source coding and may be incomplete or approximate.
- Events may be related in time and space, limiting independence assumptions.
- Results describe the downloaded historical snapshot, not current conditions.

### Reproducibility checklist
- [ ] Run from the project folder or set `DATA_DIR` to the Phase 2 processed CSV folder.
- [ ] Confirm the shared input-check table before interpreting results.
- [ ] Keep event, week, district-week, and coordinate units distinct.
- [ ] Retain missing-district records in Gaza-wide totals; omit only from district-level analysis.
- [ ] Apply coast exclusion only to the land focal-incident table.
- [ ] Mark the incomplete final week and use prior-week features only for historical predictors.
- [ ] Run top-to-bottom and save refreshed outputs before submission.

### Sources
1. DS312, *Final Project Guide*, Phase 3: Exploratory Data Analysis.
2. Group 7, `DMA_Phase1.ipynb`, business questions and proposed methods.
3. Group 7, `DMA_Phase2.ipynb`, preprocessing decisions and processed CSV definitions.
4. Group 7 Phase 3 contributions: `DMA_Phase3_Bivariate_Multivariate.ipynb`, `DMA_Phase3_Hypothesis_Testing.ipynb`, `DMA_Phase3_Pattern_Spotting.ipynb`, and `EDA_Phase3_AnsweringBQ-1.ipynb`.
5. ACLED Codebook and export documentation, as cited in Phase 1 and Phase 2.